# Complete HWG calculation pipeline: an auditable stored-evidence walkthrough

**Theory:** `su3_nf5_k3o2_infinite` · **cutoff:** $t^{10}$ · **scope:** manifest $SU(5)$ branching

`EXECUTION_MODE = "stored-results"`

This learning document and dissertation audit trail loads completed physical results. It **does not recompute** expansion, restoration, PL, reconstruction, operator analysis, branching, charge maps, or blind benchmarks. Its optional code cells perform only lightweight loading, exact-arithmetic toys, and representation conventions. Git dates below record when an evidence version entered Git, not necessarily the calculation's wall-clock start.

## 0. Notebook purpose and reading guide

Calculation means a stored physical result; checking means comparison with an invariant or independent route; interpretation means a conservative physical reading and is labelled as such. Stored machine checks, notebook re-evaluations, and external/user-reported comparisons are distinct.

### Table of contents
- [0. Purpose and reading guide](#0-purpose-and-reading-guide)
- [1. Whole-pipeline overview](#1-whole-pipeline-overview)
- [2. Project and repository layout](#2-project-and-repository-layout)
- [3. Stage 0 — Source HWG and structured input](#3-stage-0-source-hwg-and-structured-input)
- [4. Stage 1 — Highest-weight expansion](#4-stage-1-highest-weight-expansion)
- [5. Stage 2 — Restoration of irreducible characters](#5-stage-2-restoration-of-irreducible-characters)
- [6. Stage 3 — Dimension refinement and unrefinement](#6-stage-3-dimension-refinement-and-unrefinement)
- [7. Stage 4 — Refined plethystic logarithm](#7-stage-4-refined-plethystic-logarithm)
- [8. Stage 5 — Plethystic reconstruction](#8-stage-5-plethystic-reconstruction)
- [9. Stage 6 — Low-degree operator analysis](#9-stage-6-low-degree-operator-analysis)
- [10. Stage 7 — Branching to the manifest subgroup](#10-stage-7-branching-to-the-manifest-subgroup)
- [11. How checks are organised](#11-how-checks-are-organised)
- [12. Independent validation benchmarks](#12-independent-validation-benchmarks)
- [13. Implementation guide](#13-implementation-guide)
- [14. How to run a new theory](#14-how-to-run-a-new-theory)
- [15. Limitations and next steps](#15-limitations-and-next-steps)
- [Appendices A–J](#appendices-a-j)

**Legend.** $[a_1,\ldots,a_r]_{A_r}$ is a Dynkin highest weight; $q$ is the original external charge; $x$ is the raw branching charge. ✅/❌ are explicit machine PASS/FAIL; ⚪ means unavailable, never an inferred pass.

## 1. Whole-pipeline overview

> **source formula → structured theory fixture → highest-weight expansion → irreducible-character restoration → q-refined/unrefined Hilbert series → refined PL → PE reconstruction → generator/relation channel analysis → SU(5) × U(1)_x branching**

Each arrow changes the stored mathematical object: YAML normalization (`io.load_theory`) → sparse monomials (`expansion.expand_hwg`) → irreducible characters (`characters.restore_characters`) → dimensions (`dimension_refine`, `unrefine`) → virtual characters (`plethystic.plethystic_logarithm`) → round trip (`plethystic_exponential`) → conservative channels (`operators.analyze_operator_content`) → restricted child characters (`branching.branch_character`).

| stage | input | operation | stored output | primary check | status |
|---|---|---|---|---|---|
| 0 | source | input | fixture/audit | stored explicit evidence | PASS |
| 1 | input | hwg | hwg_expansion.json, checks.json | stored explicit evidence | PASS |
| 2 | hwg | characters | character_series.json, character_checks.json | stored explicit evidence | PASS |
| 3 | characters | dimensions | q_refined_dimension_series.json, unrefined_hilbert_series.json | stored explicit evidence | PASS |
| 4 | dimensions | plethystic-log | refined_plethystic_logarithm.json, q_refined_dimension_pl.json, unrefined_plethystic_logarithm.json, plethystic_logarithm_checks.json | stored explicit evidence | PASS |
| 5 | plethystic-log | reconstruction | reconstructed_character_series.json, reconstructed_q_refined_dimension_series.json, reconstructed_unrefined_hilbert_series.json, reconstruction_difference.json, reconstruction_checks.json | stored explicit evidence | PASS |
| 6 | reconstruction | operator-analysis | operator_content.json, candidate_generators.json, first_relation_candidates.json, first_relation_channels.json, operator_content_checks.json | stored explicit evidence | PASS |
| 7 | operator-analysis | branching | manifest_branching/branched_character_series.json, manifest_branching/branched_refined_plethystic_logarithm.json, manifest_branching/branched_candidate_generators.json, manifest_branching/branched_first_relation_candidates.json, manifest_branching/branching_checks.json | stored explicit evidence | PASS |

## 2. Project and repository layout

- `theories/`: authoritative structured exact inputs and branching embeddings.
- `references/overleaf/`: copied source LaTeX, never parsed for data.
- `src/hwg_pipeline/`: generic exact algorithms and this report builder.
- `tests/`: mathematical conventions, properties, regressions, and report tests.
- `generated/`: immutable physical evidence plus generated reports.
- `notebooks/`: readable entry points; no essential implementation.
- `scripts/`: repository Sage/Sage-Python launchers.

|Module|Public responsibility|Stages|
|---|---|---|
|`io.py` / `model.py`|schema loading and exact domain objects|input|
|`expansion.py`|sparse truncated PE/product expansion|HWG|
|`characters.py`|Weyl characters and dimensions|characters, dimensions|
|`plethystic.py`|formal log/exp, Adams, PL/PE|PL, reconstruction|
|`operators.py`|candidate generators and first relations|operator analysis|
|`branching.py`|weight restriction and child reconstruction|branching|
|`notebook_report.py`|read-only evidence normalization/rendering|report|

## 3. Stage 0 — Source HWG and structured input

| Stage field | Audit record |
|---|---|
| Stage | 0 — input |
| Mathematical input | source formula and theory metadata |
| Mathematical operation | schema-normalize exact PE terms |
| Mathematical output | validated theory fixture |
| Code module | `hwg_pipeline.io` |
| Principal function/class | `load_theory` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline ...` |
| Primary stored evidence | `theories/su3_nf5_k3o2_infinite.yaml` |
| Checks / validation target | schema, exact rationals, factor declarations, labels, source preservation |
| Evidence Git provenance | `a39d9050c7f85c5678b4080eeebbf4c59a63975c` (2026-07-23T15:19:55+01:00; Add SU(3)+5F infinite-coupling fixture) |
| Stage status | ✅ **PASS** |

The theory is five-dimensional SU(3) with five flavours, $|k|=3/2$, infinite coupling, enhanced $SU(6)\times U(1)_q$. $t$ grades the ring; $\mu_i$ record **highest weights**, not weights inside a character.

$$\operatorname{PE}[(\mu_1\mu_5+1)t^2+(q\mu_2+q^{-1}\mu_4)t^3+\mu_2\mu_4t^4-\mu_2\mu_4t^6].$$

The structured rational product is preserved in the YAML and is the independent expansion route; no TeX is parsed.

| coefficient | degree | A5 labels | q | meaning |
|---|---|---|---|---|
| 1 | 2 | [1,0,0,0,1]_{A5} | 0 | highest-weight monomial |
| 1 | 2 | [0,0,0,0,0]_{A5} | 0 | highest-weight monomial |
| 1 | 3 | [0,1,0,0,0]_{A5} | 1 | highest-weight monomial |
| 1 | 3 | [0,0,0,1,0]_{A5} | -1 | highest-weight monomial |
| 1 | 4 | [0,1,0,1,0]_{A5} | 0 | highest-weight monomial |
| -1 | 6 | [0,1,0,1,0]_{A5} | 0 | highest-weight monomial |

In [ ]:
from pathlib import Path
from sage.all import QQ
from hwg_pipeline.io import load_theory
ROOT = Path.cwd().resolve()
while not (ROOT / 'theories').is_dir(): ROOT = ROOT.parent
theory = load_theory(ROOT/'theories/su3_nf5_k3o2_infinite.yaml')
print(theory.id, QQ(3)/QQ(2), theory.simple_factors, theory.pe_terms)

## 4. Stage 1 — Highest-weight expansion

| Stage field | Audit record |
|---|---|
| Stage | 1 — Highest-weight expansion |
| Mathematical input | structured PE/rational product |
| Mathematical operation | truncated sparse product; degrees, Dynkin exponents, charges add |
| Mathematical output | highest-weight series |
| Code module | `hwg_pipeline.expansion` |
| Principal function/class | `expand_pe`, `expand_rational_product` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline expand su3_nf5_k3o2_infinite --order 10` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json` |
| Checks / validation target | PE route versus rational-product route; integrality, positivity, cutoff, stability |
| Evidence Git provenance | `3d3a7bae82c586b7363ada08221758bb90f4f168` (2026-07-23T15:28:16+01:00; Implement exact sparse HWG expansion) |
| Stage status | ✅ **PASS** |

For $\operatorname{PE}[\sum_a c_aM_a]=\prod_a(1-M_a)^{-c_a}$, sparse keys contain degree, label exponents and exact charges. Multiplication adds these data. **This is not representation tensor-product multiplication.** The complete stored physical result below was loaded, not recomputed.

<details><summary>Complete stored terms through the cutoff</summary>

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 3 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 3 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 4 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 4 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 4 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 4 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 5 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 5 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 5 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 5 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 6 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 6 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 6 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 6 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 6 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 6 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 6 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 6 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 7 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 7 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 7 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 7 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 7 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 7 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 8 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 8 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 8 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 8 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 8 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 8 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 8 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 8 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 8 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 8 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 8 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 9 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,0,0,3,0]_{A5} | 1 | 0 | -3 |
| 9 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 9 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 9 | [0,3,0,0,0]_{A5} | 1 | 0 | 3 |
| 9 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 9 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 9 | [1,1,0,2,1]_{A5} | 1 | 0 | -1 |
| 9 | [1,2,0,1,1]_{A5} | 1 | 0 | 1 |
| 9 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 9 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 9 | [3,0,0,1,3]_{A5} | 1 | 0 | -1 |
| 9 | [3,1,0,0,3]_{A5} | 1 | 0 | 1 |
| 10 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 10 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,1,0,3,0]_{A5} | 1 | 0 | -2 |
| 10 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 10 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,3,0,1,0]_{A5} | 1 | 0 | 2 |
| 10 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 10 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 10 | [1,2,0,2,1]_{A5} | 1 | 0 | 0 |
| 10 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 10 | [2,0,0,2,2]_{A5} | 1 | 0 | -2 |
| 10 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 10 | [2,2,0,0,2]_{A5} | 1 | 0 | 2 |
| 10 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 10 | [3,1,0,1,3]_{A5} | 1 | 0 | 0 |
| 10 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 10 | [5,0,0,0,5]_{A5} | 1 | 0 | 0 |

</details>

In [ ]:
from sage.all import PowerSeriesRing, QQ
R = PowerSeriesRing(QQ, 't', default_prec=12); t=R.gen()
assert ((1-t**4)/(1-t**2)) == 1+t**2
print('PE[t^2-t^4] =', (1-t**4)/(1-t**2))

## 5. Stage 2 — Restoration of irreducible characters

| Stage field | Audit record |
|---|---|
| Stage | 2 — Restoration of irreducible characters |
| Mathematical input | highest-weight monomials |
| Mathematical operation | map each label tuple to one Weyl irreducible character |
| Mathematical output | character Hilbert series |
| Code module | `hwg_pipeline.characters` |
| Principal function/class | `restore_characters` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline characters su3_nf5_k3o2_infinite --order 10` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/character_series.json` |
| Checks / validation target | degree/charge preservation, dimensions, serialization |
| Evidence Git provenance | `d562e55b97222ee53371097c7a3b1818f5550a22` (2026-07-23T15:50:21+01:00; Verify character pipeline with Sage launcher) |
| Stage status | ✅ **PASS** |

A highest-weight fugacity is only an irrep label: $\mu_1\mu_5\mapsto[1,0,0,0,1]_{A_5}$ and $\mu_1^2\mu_5^2\mapsto[2,0,0,0,2]_{A_5}$, **not** the adjoint tensor square. `WeylCharacterRing('A5', style='coroots')` matches Dynkin-label coordinates.

<details><summary>Complete stored terms through the cutoff</summary>

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 3 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 3 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 4 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 4 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 4 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 4 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 5 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 5 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 5 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 5 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 6 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 6 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 6 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 6 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 6 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 6 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 6 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 6 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 7 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 7 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 7 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 7 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 7 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 7 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 7 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 8 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 8 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 8 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 8 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 8 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 8 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 8 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 8 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 8 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 8 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 8 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 9 | [0,0,0,3,0]_{A5} | 1 | 0 | -3 |
| 9 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 9 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 9 | [1,1,0,2,1]_{A5} | 1 | 0 | -1 |
| 9 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 9 | [3,0,0,1,3]_{A5} | 1 | 0 | -1 |
| 9 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 9 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 9 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 9 | [1,2,0,1,1]_{A5} | 1 | 0 | 1 |
| 9 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 9 | [3,1,0,0,3]_{A5} | 1 | 0 | 1 |
| 9 | [0,3,0,0,0]_{A5} | 1 | 0 | 3 |
| 10 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 10 | [0,1,0,3,0]_{A5} | 1 | 0 | -2 |
| 10 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 10 | [2,0,0,2,2]_{A5} | 1 | 0 | -2 |
| 10 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 10 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,2,0,2,1]_{A5} | 1 | 0 | 0 |
| 10 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 10 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 10 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 10 | [3,1,0,1,3]_{A5} | 1 | 0 | 0 |
| 10 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 10 | [5,0,0,0,5]_{A5} | 1 | 0 | 0 |
| 10 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 10 | [0,3,0,1,0]_{A5} | 1 | 0 | 2 |
| 10 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 10 | [2,2,0,0,2]_{A5} | 1 | 0 | 2 |

</details>

In [ ]:
from sage.all import WeylCharacterRing
A2=WeylCharacterRing('A2',style='coroots'); A5=WeylCharacterRing('A5',style='coroots')
assert [A2(1,1).degree(),A5(1,0,0,0,1).degree(),A5(0,1,0,1,0).degree(),A5(2,0,0,0,2).degree()]==[8,35,189,405]
assert A5(1,0,0,0,0)*A5(0,0,0,0,1)==A5(0,0,0,0,0)+A5(1,0,0,0,1)
print('Sage coroot/Dynkin convention checks passed')

## 6. Stage 3 — Dimension refinement and unrefinement

| Stage field | Audit record |
|---|---|
| Stage | 3 — Dimension refinement and unrefinement |
| Mathematical input | character Hilbert series |
| Mathematical operation | evaluate dimensions, preserve q, then sum q sectors |
| Mathematical output | scalar refined/unrefined series |
| Code module | `hwg_pipeline.characters` |
| Principal function/class | `dimension_refine`, `unrefine` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline characters su3_nf5_k3o2_infinite --order 10` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/q_refined_dimension_series.json` |
| Checks / validation target | exact integer dimensions; q=1 equality |
| Evidence Git provenance | `d562e55b97222ee53371097c7a3b1818f5550a22` (2026-07-23T15:50:21+01:00; Verify character pipeline with Sage launcher) |
| Stage status | ✅ **PASS** |

Dimension evaluation replaces each character by its Weyl dimension while retaining q. Unrefinement sets q=1 by summing charge sectors. Hand checks: $1+35=36$, $15+15=30$, and $1+35+405+189=630$.

<details><summary>Complete stored terms through the cutoff</summary>

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | — | 1 | 0 | 0 |
| 2 | — | 36 | 0 | 0 |
| 3 | — | 15 | 0 | -1 |
| 3 | — | 15 | 0 | 1 |
| 4 | — | 630 | 0 | 0 |
| 5 | — | 399 | 0 | -1 |
| 5 | — | 399 | 0 | 1 |
| 6 | — | 105 | 0 | -2 |
| 6 | — | 7000 | 0 | 0 |
| 6 | — | 105 | 0 | 2 |
| 7 | — | 5250 | 0 | -1 |
| 7 | — | 5250 | 0 | 1 |
| 8 | — | 2310 | 0 | -2 |
| 8 | — | 56160 | 0 | 0 |
| 8 | — | 2310 | 0 | 2 |
| 9 | — | 490 | 0 | -3 |
| 9 | — | 45954 | 0 | -1 |
| 9 | — | 45954 | 0 | 1 |
| 9 | — | 490 | 0 | 3 |
| 10 | — | 25830 | 0 | -2 |
| 10 | — | 352134 | 0 | 0 |
| 10 | — | 25830 | 0 | 2 |

</details>

### Complete unrefined series
| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | — | 1 | 0 | 0 |
| 1 | — | 0 | 0 | 0 |
| 2 | — | 36 | 0 | 0 |
| 3 | — | 30 | 0 | 0 |
| 4 | — | 630 | 0 | 0 |
| 5 | — | 798 | 0 | 0 |
| 6 | — | 7210 | 0 | 0 |
| 7 | — | 10500 | 0 | 0 |
| 8 | — | 60780 | 0 | 0 |
| 9 | — | 92888 | 0 | 0 |
| 10 | — | 403794 | 0 | 0 |

In [ ]:
assert (1+35,15+15,1+35+405+189)==(36,30,630)
print('exact low-degree dimension sums passed')

## 7. Stage 4 — Refined plethystic logarithm

| Stage field | Audit record |
|---|---|
| Stage | 4 — Refined plethystic logarithm |
| Mathematical input | character Hilbert series |
| Mathematical operation | Möbius-weighted formal logs of Adams transforms |
| Mathematical output | signed virtual-character PL |
| Code module | `hwg_pipeline.plethystic` |
| Principal function/class | `plethystic_logarithm` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline plethystic-log su3_nf5_k3o2_infinite --order 10` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/refined_plethystic_logarithm.json` |
| Checks / validation target | Adams convention; scalar independent route; integrality; stability |
| Evidence Git provenance | `ebc32f70bed45f167dd50eb67152a646d3c0185d` (2026-07-23T16:14:39+01:00; Implement exact refined plethystic logarithms) |
| Stage status | ✅ **PASS** |

$$\operatorname{PL}[H]=\sum_{k\ge1}\frac{\mu(k)}k\log(\psi_k(H)).$$ Ordinary formal log supplies rational intermediate coefficients; Möbius inversion removes multicover contributions. Adams $\psi_k$ scales t-degree and q charge and calls `character.adams_operator(k)` on characters—it is **not Dynkin-label scaling**. Signed virtual representations are retained. Hence this character-valued PL differs from the short highest-weight PE exponent.

<details><summary>Complete stored terms through the cutoff</summary>

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 2 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 3 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 3 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 4 | [0,0,0,0,0]_{A5} | -1 | 0 | 0 |
| 4 | [1,0,0,0,1]_{A5} | -1 | 0 | 0 |
| 5 | [0,0,0,0,2]_{A5} | -1 | 0 | -1 |
| 5 | [0,0,0,1,0]_{A5} | -1 | 0 | -1 |
| 5 | [1,0,1,0,0]_{A5} | -1 | 0 | -1 |
| 5 | [0,0,1,0,1]_{A5} | -1 | 0 | 1 |
| 5 | [0,1,0,0,0]_{A5} | -1 | 0 | 1 |
| 5 | [2,0,0,0,0]_{A5} | -1 | 0 | 1 |
| 6 | [0,1,0,0,0]_{A5} | -1 | 0 | -2 |
| 6 | [0,0,0,0,0]_{A5} | -1 | 0 | 0 |
| 6 | [0,0,2,0,0]_{A5} | -1 | 0 | 0 |
| 6 | [0,1,0,1,0]_{A5} | -1 | 0 | 0 |
| 6 | [0,0,0,1,0]_{A5} | -1 | 0 | 2 |
| 7 | [0,0,0,0,2]_{A5} | 2 | 0 | -1 |
| 7 | [0,0,0,1,0]_{A5} | 2 | 0 | -1 |
| 7 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 7 | [1,0,1,0,0]_{A5} | 2 | 0 | -1 |
| 7 | [2,1,0,0,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,0,0,1,2]_{A5} | 1 | 0 | 1 |
| 7 | [0,0,1,0,1]_{A5} | 2 | 0 | 1 |
| 7 | [0,1,0,0,0]_{A5} | 2 | 0 | 1 |
| 7 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 7 | [2,0,0,0,0]_{A5} | 2 | 0 | 1 |
| 8 | [0,0,1,0,1]_{A5} | 2 | 0 | -2 |
| 8 | [0,1,0,0,0]_{A5} | 2 | 0 | -2 |
| 8 | [1,1,0,0,1]_{A5} | 1 | 0 | -2 |
| 8 | [2,0,0,0,0]_{A5} | 1 | 0 | -2 |
| 8 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,0,1,1,1]_{A5} | 2 | 0 | 0 |
| 8 | [0,0,2,0,0]_{A5} | 4 | 0 | 0 |
| 8 | [0,1,0,0,2]_{A5} | 2 | 0 | 0 |
| 8 | [0,1,0,1,0]_{A5} | 5 | 0 | 0 |
| 8 | [1,0,0,0,1]_{A5} | 5 | 0 | 0 |
| 8 | [1,1,1,0,0]_{A5} | 2 | 0 | 0 |
| 8 | [2,0,0,1,0]_{A5} | 2 | 0 | 0 |
| 8 | [0,0,0,0,2]_{A5} | 1 | 0 | 2 |
| 8 | [0,0,0,1,0]_{A5} | 2 | 0 | 2 |
| 8 | [1,0,0,1,1]_{A5} | 1 | 0 | 2 |
| 8 | [1,0,1,0,0]_{A5} | 2 | 0 | 2 |
| 9 | [1,0,0,0,1]_{A5} | 1 | 0 | -3 |
| 9 | [0,0,0,0,2]_{A5} | -2 | 0 | -1 |
| 9 | [0,0,0,1,0]_{A5} | -3 | 0 | -1 |
| 9 | [0,0,2,1,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,1,1,0,1]_{A5} | 2 | 0 | -1 |
| 9 | [1,0,0,0,3]_{A5} | -1 | 0 | -1 |
| 9 | [1,0,0,1,1]_{A5} | -2 | 0 | -1 |
| 9 | [1,0,1,0,0]_{A5} | -1 | 0 | -1 |
| 9 | [2,0,1,0,1]_{A5} | -1 | 0 | -1 |
| 9 | [2,1,0,0,0]_{A5} | -2 | 0 | -1 |
| 9 | [4,0,0,0,0]_{A5} | -1 | 0 | -1 |
| 9 | [0,0,0,0,4]_{A5} | -1 | 0 | 1 |
| 9 | [0,0,0,1,2]_{A5} | -2 | 0 | 1 |
| 9 | [0,0,1,0,1]_{A5} | -1 | 0 | 1 |
| 9 | [0,1,0,0,0]_{A5} | -3 | 0 | 1 |
| 9 | [0,1,2,0,0]_{A5} | 1 | 0 | 1 |
| 9 | [1,0,1,0,2]_{A5} | -1 | 0 | 1 |
| 9 | [1,0,1,1,0]_{A5} | 2 | 0 | 1 |
| 9 | [1,1,0,0,1]_{A5} | -2 | 0 | 1 |
| 9 | [2,0,0,0,0]_{A5} | -2 | 0 | 1 |
| 9 | [3,0,0,0,1]_{A5} | -1 | 0 | 1 |
| 9 | [1,0,0,0,1]_{A5} | 1 | 0 | 3 |
| 10 | [0,0,0,1,2]_{A5} | -2 | 0 | -2 |
| 10 | [0,0,0,2,0]_{A5} | -1 | 0 | -2 |
| 10 | [0,0,1,0,1]_{A5} | -8 | 0 | -2 |
| 10 | [0,1,0,0,0]_{A5} | -6 | 0 | -2 |
| 10 | [1,0,1,0,2]_{A5} | -1 | 0 | -2 |
| 10 | [1,0,1,1,0]_{A5} | -2 | 0 | -2 |
| 10 | [1,1,0,0,1]_{A5} | -6 | 0 | -2 |
| 10 | [2,0,0,0,0]_{A5} | -5 | 0 | -2 |
| 10 | [2,1,0,1,0]_{A5} | -1 | 0 | -2 |
| 10 | [3,0,0,0,1]_{A5} | -1 | 0 | -2 |
| 10 | [0,0,0,0,0]_{A5} | -7 | 0 | 0 |
| 10 | [0,0,0,2,2]_{A5} | -1 | 0 | 0 |
| 10 | [0,0,0,3,0]_{A5} | -1 | 0 | 0 |
| 10 | [0,0,1,0,3]_{A5} | -3 | 0 | 0 |
| 10 | [0,0,1,1,1]_{A5} | -11 | 0 | 0 |
| 10 | [0,0,2,0,0]_{A5} | -10 | 0 | 0 |
| 10 | [0,1,0,0,2]_{A5} | -12 | 0 | 0 |
| 10 | [0,1,0,1,0]_{A5} | -21 | 0 | 0 |
| 10 | [0,3,0,0,0]_{A5} | -1 | 0 | 0 |
| 10 | [1,0,0,0,1]_{A5} | -21 | 0 | 0 |
| 10 | [1,0,2,0,1]_{A5} | -2 | 0 | 0 |
| 10 | [1,1,0,1,1]_{A5} | -4 | 0 | 0 |
| 10 | [1,1,1,0,0]_{A5} | -11 | 0 | 0 |
| 10 | [2,0,0,0,2]_{A5} | -4 | 0 | 0 |
| 10 | [2,0,0,1,0]_{A5} | -12 | 0 | 0 |
| 10 | [2,2,0,0,0]_{A5} | -1 | 0 | 0 |
| 10 | [3,0,1,0,0]_{A5} | -3 | 0 | 0 |
| 10 | [0,0,0,0,2]_{A5} | -5 | 0 | 2 |
| 10 | [0,0,0,1,0]_{A5} | -6 | 0 | 2 |
| 10 | [0,1,0,1,2]_{A5} | -1 | 0 | 2 |
| 10 | [0,1,1,0,1]_{A5} | -2 | 0 | 2 |
| 10 | [0,2,0,0,0]_{A5} | -1 | 0 | 2 |
| 10 | [1,0,0,0,3]_{A5} | -1 | 0 | 2 |
| 10 | [1,0,0,1,1]_{A5} | -6 | 0 | 2 |
| 10 | [1,0,1,0,0]_{A5} | -8 | 0 | 2 |
| 10 | [2,0,1,0,1]_{A5} | -1 | 0 | 2 |
| 10 | [2,1,0,0,0]_{A5} | -2 | 0 | 2 |

</details>

### `q_refined_dimension_pl.json`
| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 2 | — | 36 | 0 | 0 |
| 3 | — | 15 | 0 | -1 |
| 3 | — | 15 | 0 | 1 |
| 4 | — | -36 | 0 | 0 |
| 5 | — | -141 | 0 | -1 |
| 5 | — | -141 | 0 | 1 |
| 6 | — | -15 | 0 | -2 |
| 6 | — | -365 | 0 | 0 |
| 6 | — | -15 | 0 | 2 |
| 7 | — | 876 | 0 | -1 |
| 7 | — | 876 | 0 | 1 |
| 8 | — | 645 | 0 | -2 |
| 8 | — | 6525 | 0 | 0 |
| 8 | — | 645 | 0 | 2 |
| 9 | — | 35 | 0 | -3 |
| 9 | — | 48 | 0 | -1 |
| 9 | — | 48 | 0 | 1 |
| 9 | — | 35 | 0 | 3 |
| 10 | — | -10410 | 0 | -2 |
| 10 | — | -65439 | 0 | 0 |
| 10 | — | -10410 | 0 | 2 |

### `unrefined_plethystic_logarithm.json`
| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 2 | — | 36 | 0 | 0 |
| 3 | — | 30 | 0 | 0 |
| 4 | — | -36 | 0 | 0 |
| 5 | — | -282 | 0 | 0 |
| 6 | — | -395 | 0 | 0 |
| 7 | — | 1752 | 0 | 0 |
| 8 | — | 7815 | 0 | 0 |
| 9 | — | 166 | 0 | 0 |
| 10 | — | -86259 | 0 | 0 |

In [ ]:
from sage.all import QQ, PowerSeriesRing, moebius
R=PowerSeriesRing(QQ,'t',default_prec=12); t=R.gen(); H=1/(1-t**2)
ordinary=H.log(); pl=sum(QQ(moebius(k))/k * (H(t=t**k)).log() for k in range(1,6))
print('ordinary log =',ordinary); print('PL =',pl); assert pl[2]==1 and all(pl[d]==0 for d in range(3,11))
assert ((1+t**2).log()-(QQ(1)/2)*(1+t**4).log())[2:6]==[1,0,-1,0]

## 8. Stage 5 — Plethystic reconstruction

| Stage field | Audit record |
|---|---|
| Stage | 5 — Plethystic reconstruction |
| Mathematical input | refined character PL |
| Mathematical operation | plethystic exponential with Adams operations |
| Mathematical output | reconstructed Hilbert series |
| Code module | `hwg_pipeline.plethystic` |
| Principal function/class | `plethystic_exponential` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline reconstruct su3_nf5_k3o2_infinite --order 10` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/reconstruction_difference.json` |
| Checks / validation target | representation/q/degree equality and independent scalar PE |
| Evidence Git provenance | `d55b1caf1e54e50083ec2a330f588913788e0104` (2026-07-23T16:28:06+01:00; Add exact plethystic reconstruction) |
| Stage status | ✅ **PASS** |

$$\operatorname{PE}[F]=\exp\!\left[\sum_{k\ge1}\psi_k(F)/kight].$$ PE has $1/k$ but no Möbius factor. Stored refined, q-refined, and independent scalar comparisons establish the round trip; no physical calculation is rerun here.

**Stored structured difference:** mismatch count `0`; list `[]`. Therefore **PE[PL[H]] = H mod t^11** is stated only because the stored equality checks pass.

## 9. Stage 6 — Low-degree operator analysis

| Stage field | Audit record |
|---|---|
| Stage | 6 — Low-degree operator analysis |
| Mathematical input | signed refined PL |
| Mathematical operation | classify positives before first negative and negatives at first negative |
| Mathematical output | candidate generators/relations |
| Code module | `hwg_pipeline.operators` |
| Principal function/class | `analyze_operator_content` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline analyze-pl su3_nf5_k3o2_infinite --order 10` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/operator_content.json` |
| Checks / validation target | first negative degree, Sym² channels, exact deficit |
| Evidence Git provenance | `7c50c0d7fa9287df82aed756d311f81dcb61c343` (2026-07-23T16:39:57+01:00; Analyze refined PL operator content) |
| Stage status | ✅ **PASS** |

**Interpretation rule (conservative):** positives before the first negative degree are candidate generators; negatives at the first negative degree are first relation candidates; later terms can mix relations, syzygies and cancellations. At degree four, $\mathrm{Sym}^2(1+35)=2(1)+2(35)+405+189$, while $H_4=1+35+405+189$, so the deficit is $1+35$ and $666-630=36$.

### Candidate generators
| degree | representation | mult. | dimension | x | q | source/classification |
|---|---|---|---|---|---|---|
| 2 | [0,0,0,0,0]_{A5} | 1 | 1 | 0 | 0 | low_degree_generator_candidate |
| 2 | [1,0,0,0,1]_{A5} | 1 | 35 | 0 | 0 | low_degree_generator_candidate |
| 3 | [0,0,0,1,0]_{A5} | 1 | 15 | 0 | -1 | low_degree_generator_candidate |
| 3 | [0,1,0,0,0]_{A5} | 1 | 15 | 0 | 1 | low_degree_generator_candidate |

### First relation candidates
| degree | representation | mult. | dimension | x | q | source/classification |
|---|---|---|---|---|---|---|
| 4 | [0,0,0,0,0]_{A5} | -1 | 1 | 0 | 0 | first_relation_candidate |
| 4 | [1,0,0,0,1]_{A5} | -1 | 35 | 0 | 0 | first_relation_candidate |

## 10. Stage 7 — Branching to manifest subgroup

| Stage field | Audit record |
|---|---|
| Stage | 7 — Branching to manifest subgroup |
| Mathematical input | SU(6) character series and PL |
| Mathematical operation | restrict weights and reconstruct SU(5) irreps by raw x charge |
| Mathematical output | SU(5) × U(1)_x × U(1)_q series |
| Code module | `hwg_pipeline.branching` |
| Principal function/class | `branch_character` |
| Original CLI (not executed here) | `./scripts/sage-python -m hwg_pipeline branch su3_nf5_k3o2_infinite --order 10 --branching su3_nf5_k3o2_to_manifest` |
| Primary stored evidence | `generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branched_character_series.json` |
| Checks / validation target | normalization, dimensions, charges, conjugation, tensor compatibility |
| Evidence Git provenance | `a49d1b010160ba3d31f70c2cdba5a6fe658bf572` (2026-07-23T17:05:04+01:00; Implement exact manifest symmetry branching) |
| Stage status | ✅ **PASS** |

The embedding is $SU(6)\to SU(5)\times U(1)_x$ with $6\to5_{+1}+1_{-5}$. The original q is preserved independently. x is a raw branching charge; neither x nor q is automatically baryon or instanton charge. The algorithm restricts parent weights, computes x, groups weights by charge, reconstructs child irreducibles, and checks dimensions.

<details><summary>Complete stored terms through the cutoff</summary>

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | [0, 0, 0, 0] | 1 | 0 | 0 |
| 2 | [0, 0, 0, 0] | 2 | 0 | 0 |
| 2 | [0, 0, 0, 1] | 1 | -6 | 0 |
| 2 | [1, 0, 0, 0] | 1 | 6 | 0 |
| 2 | [1, 0, 0, 1] | 1 | 0 | 0 |
| 3 | [0, 0, 0, 1] | 1 | 4 | -1 |
| 3 | [0, 0, 1, 0] | 1 | -2 | -1 |
| 3 | [0, 1, 0, 0] | 1 | 2 | 1 |
| 3 | [1, 0, 0, 0] | 1 | -4 | 1 |
| 4 | [0, 0, 0, 0] | 3 | 0 | 0 |
| 4 | [0, 0, 0, 1] | 2 | -6 | 0 |
| 4 | [0, 0, 0, 2] | 1 | -12 | 0 |
| 4 | [0, 1, 0, 1] | 1 | 6 | 0 |
| 4 | [0, 1, 1, 0] | 1 | 0 | 0 |
| 4 | [1, 0, 0, 0] | 2 | 6 | 0 |
| 4 | [1, 0, 0, 1] | 3 | 0 | 0 |
| 4 | [1, 0, 0, 2] | 1 | -6 | 0 |
| 4 | [1, 0, 1, 0] | 1 | -6 | 0 |
| 4 | [2, 0, 0, 0] | 1 | 12 | 0 |
| 4 | [2, 0, 0, 1] | 1 | 6 | 0 |
| 4 | [2, 0, 0, 2] | 1 | 0 | 0 |
| 5 | [0, 0, 0, 1] | 2 | 4 | -1 |
| 5 | [0, 0, 0, 2] | 1 | -2 | -1 |
| 5 | [0, 0, 1, 0] | 2 | -2 | -1 |
| 5 | [0, 0, 1, 1] | 1 | -8 | -1 |
| 5 | [0, 1, 0, 0] | 2 | 2 | 1 |
| 5 | [0, 1, 0, 1] | 1 | -4 | 1 |
| 5 | [1, 0, 0, 0] | 2 | -4 | 1 |
| 5 | [1, 0, 0, 1] | 1 | -10 | 1 |
| 5 | [1, 0, 0, 1] | 1 | 10 | -1 |
| 5 | [1, 0, 0, 2] | 1 | 4 | -1 |
| 5 | [1, 0, 1, 0] | 1 | 4 | -1 |
| 5 | [1, 0, 1, 1] | 1 | -2 | -1 |
| 5 | [1, 1, 0, 0] | 1 | 8 | 1 |
| 5 | [1, 1, 0, 1] | 1 | 2 | 1 |
| 5 | [2, 0, 0, 0] | 1 | 2 | 1 |
| 5 | [2, 0, 0, 1] | 1 | -4 | 1 |
| 6 | [0, 0, 0, 0] | 4 | 0 | 0 |
| 6 | [0, 0, 0, 1] | 3 | -6 | 0 |
| 6 | [0, 0, 0, 2] | 2 | -12 | 0 |
| 6 | [0, 0, 0, 2] | 1 | 8 | -2 |
| 6 | [0, 0, 0, 3] | 1 | -18 | 0 |
| 6 | [0, 0, 1, 1] | 1 | 2 | -2 |
| 6 | [0, 0, 2, 0] | 1 | -4 | -2 |
| 6 | [0, 1, 0, 1] | 2 | 6 | 0 |
| 6 | [0, 1, 0, 2] | 1 | 0 | 0 |
| 6 | [0, 1, 1, 0] | 2 | 0 | 0 |
| 6 | [0, 1, 1, 1] | 1 | -6 | 0 |
| 6 | [0, 2, 0, 0] | 1 | 4 | 2 |
| 6 | [1, 0, 0, 0] | 3 | 6 | 0 |
| 6 | [1, 0, 0, 1] | 5 | 0 | 0 |
| 6 | [1, 0, 0, 2] | 3 | -6 | 0 |
| 6 | [1, 0, 0, 3] | 1 | -12 | 0 |
| 6 | [1, 0, 1, 0] | 2 | -6 | 0 |
| 6 | [1, 0, 1, 1] | 1 | -12 | 0 |
| 6 | [1, 1, 0, 0] | 1 | -2 | 2 |
| 6 | [1, 1, 0, 1] | 1 | 12 | 0 |
| 6 | [1, 1, 0, 2] | 1 | 6 | 0 |
| 6 | [1, 1, 1, 0] | 1 | 6 | 0 |
| 6 | [1, 1, 1, 1] | 1 | 0 | 0 |
| 6 | [2, 0, 0, 0] | 1 | -8 | 2 |
| 6 | [2, 0, 0, 0] | 2 | 12 | 0 |
| 6 | [2, 0, 0, 1] | 3 | 6 | 0 |
| 6 | [2, 0, 0, 2] | 3 | 0 | 0 |
| 6 | [2, 0, 0, 3] | 1 | -6 | 0 |
| 6 | [2, 0, 1, 0] | 1 | 0 | 0 |
| 6 | [2, 0, 1, 1] | 1 | -6 | 0 |
| 6 | [3, 0, 0, 0] | 1 | 18 | 0 |
| 6 | [3, 0, 0, 1] | 1 | 12 | 0 |
| 6 | [3, 0, 0, 2] | 1 | 6 | 0 |
| 6 | [3, 0, 0, 3] | 1 | 0 | 0 |
| 7 | [0, 0, 0, 1] | 3 | 4 | -1 |
| 7 | [0, 0, 0, 2] | 2 | -2 | -1 |
| 7 | [0, 0, 0, 3] | 1 | -8 | -1 |
| 7 | [0, 0, 1, 0] | 3 | -2 | -1 |
| 7 | [0, 0, 1, 1] | 2 | -8 | -1 |
| 7 | [0, 0, 1, 2] | 1 | -14 | -1 |
| 7 | [0, 1, 0, 0] | 3 | 2 | 1 |
| 7 | [0, 1, 0, 1] | 2 | -4 | 1 |
| 7 | [0, 1, 0, 2] | 1 | -10 | 1 |
| 7 | [0, 1, 0, 2] | 1 | 10 | -1 |
| 7 | [0, 1, 1, 1] | 1 | 4 | -1 |
| 7 | [0, 1, 2, 0] | 1 | -2 | -1 |
| 7 | [0, 2, 0, 1] | 1 | 8 | 1 |
| 7 | [0, 2, 1, 0] | 1 | 2 | 1 |
| 7 | [1, 0, 0, 0] | 3 | -4 | 1 |
| 7 | [1, 0, 0, 1] | 2 | -10 | 1 |
| 7 | [1, 0, 0, 1] | 2 | 10 | -1 |
| 7 | [1, 0, 0, 2] | 1 | -16 | 1 |
| 7 | [1, 0, 0, 2] | 3 | 4 | -1 |
| 7 | [1, 0, 0, 3] | 1 | -2 | -1 |
| 7 | [1, 0, 1, 0] | 2 | 4 | -1 |
| 7 | [1, 0, 1, 1] | 3 | -2 | -1 |
| 7 | [1, 0, 1, 2] | 1 | -8 | -1 |
| 7 | [1, 0, 2, 0] | 1 | -8 | -1 |
| 7 | [1, 1, 0, 0] | 2 | 8 | 1 |
| 7 | [1, 1, 0, 1] | 3 | 2 | 1 |
| 7 | [1, 1, 0, 2] | 1 | -4 | 1 |
| 7 | [1, 1, 1, 0] | 1 | -4 | 1 |
| 7 | [2, 0, 0, 0] | 2 | 2 | 1 |
| 7 | [2, 0, 0, 1] | 3 | -4 | 1 |
| 7 | [2, 0, 0, 1] | 1 | 16 | -1 |
| 7 | [2, 0, 0, 2] | 1 | -10 | 1 |
| 7 | [2, 0, 0, 2] | 1 | 10 | -1 |
| 7 | [2, 0, 0, 3] | 1 | 4 | -1 |
| 7 | [2, 0, 1, 0] | 1 | -10 | 1 |
| 7 | [2, 0, 1, 0] | 1 | 10 | -1 |
| 7 | [2, 0, 1, 1] | 1 | 4 | -1 |
| 7 | [2, 0, 1, 2] | 1 | -2 | -1 |
| 7 | [2, 1, 0, 0] | 1 | 14 | 1 |
| 7 | [2, 1, 0, 1] | 1 | 8 | 1 |
| 7 | [2, 1, 0, 2] | 1 | 2 | 1 |
| 7 | [3, 0, 0, 0] | 1 | 8 | 1 |
| 7 | [3, 0, 0, 1] | 1 | 2 | 1 |
| 7 | [3, 0, 0, 2] | 1 | -4 | 1 |
| 8 | [0, 0, 0, 0] | 5 | 0 | 0 |
| 8 | [0, 0, 0, 1] | 4 | -6 | 0 |
| 8 | [0, 0, 0, 2] | 3 | -12 | 0 |
| 8 | [0, 0, 0, 2] | 2 | 8 | -2 |
| 8 | [0, 0, 0, 3] | 2 | -18 | 0 |
| 8 | [0, 0, 0, 3] | 1 | 2 | -2 |
| 8 | [0, 0, 0, 4] | 1 | -24 | 0 |
| 8 | [0, 0, 1, 1] | 2 | 2 | -2 |
| 8 | [0, 0, 1, 2] | 1 | -4 | -2 |
| 8 | [0, 0, 2, 0] | 2 | -4 | -2 |
| 8 | [0, 0, 2, 1] | 1 | -10 | -2 |
| 8 | [0, 1, 0, 1] | 3 | 6 | 0 |
| 8 | [0, 1, 0, 2] | 2 | 0 | 0 |
| 8 | [0, 1, 0, 3] | 1 | -6 | 0 |
| 8 | [0, 1, 1, 0] | 3 | 0 | 0 |
| 8 | [0, 1, 1, 1] | 2 | -6 | 0 |
| 8 | [0, 1, 1, 2] | 1 | -12 | 0 |
| 8 | [0, 2, 0, 0] | 2 | 4 | 2 |
| 8 | [0, 2, 0, 1] | 1 | -2 | 2 |
| 8 | [0, 2, 0, 2] | 1 | 12 | 0 |
| 8 | [0, 2, 1, 1] | 1 | 6 | 0 |
| 8 | [0, 2, 2, 0] | 1 | 0 | 0 |
| 8 | [1, 0, 0, 0] | 4 | 6 | 0 |
| 8 | [1, 0, 0, 1] | 7 | 0 | 0 |
| 8 | [1, 0, 0, 2] | 5 | -6 | 0 |
| 8 | [1, 0, 0, 2] | 1 | 14 | -2 |
| 8 | [1, 0, 0, 3] | 3 | -12 | 0 |
| 8 | [1, 0, 0, 3] | 1 | 8 | -2 |
| 8 | [1, 0, 0, 4] | 1 | -18 | 0 |
| 8 | [1, 0, 1, 0] | 3 | -6 | 0 |
| 8 | [1, 0, 1, 1] | 2 | -12 | 0 |
| 8 | [1, 0, 1, 1] | 1 | 8 | -2 |
| 8 | [1, 0, 1, 2] | 1 | -18 | 0 |
| 8 | [1, 0, 1, 2] | 1 | 2 | -2 |
| 8 | [1, 0, 2, 0] | 1 | 2 | -2 |
| 8 | [1, 0, 2, 1] | 1 | -4 | -2 |
| 8 | [1, 1, 0, 0] | 2 | -2 | 2 |
| 8 | [1, 1, 0, 1] | 1 | -8 | 2 |
| 8 | [1, 1, 0, 1] | 2 | 12 | 0 |
| 8 | [1, 1, 0, 2] | 3 | 6 | 0 |
| 8 | [1, 1, 0, 3] | 1 | 0 | 0 |
| 8 | [1, 1, 1, 0] | 2 | 6 | 0 |
| 8 | [1, 1, 1, 1] | 3 | 0 | 0 |
| 8 | [1, 1, 1, 2] | 1 | -6 | 0 |
| 8 | [1, 1, 2, 0] | 1 | -6 | 0 |
| 8 | [1, 2, 0, 0] | 1 | 10 | 2 |
| 8 | [1, 2, 0, 1] | 1 | 4 | 2 |
| 8 | [2, 0, 0, 0] | 2 | -8 | 2 |
| 8 | [2, 0, 0, 0] | 3 | 12 | 0 |
| 8 | [2, 0, 0, 1] | 1 | -14 | 2 |
| 8 | [2, 0, 0, 1] | 5 | 6 | 0 |
| 8 | [2, 0, 0, 2] | 6 | 0 | 0 |
| 8 | [2, 0, 0, 3] | 3 | -6 | 0 |
| 8 | [2, 0, 0, 4] | 1 | -12 | 0 |
| 8 | [2, 0, 1, 0] | 2 | 0 | 0 |
| 8 | [2, 0, 1, 1] | 3 | -6 | 0 |
| 8 | [2, 0, 1, 2] | 1 | -12 | 0 |
| 8 | [2, 0, 2, 0] | 1 | -12 | 0 |
| 8 | [2, 1, 0, 0] | 1 | 4 | 2 |
| 8 | [2, 1, 0, 1] | 1 | -2 | 2 |
| 8 | [2, 1, 0, 1] | 1 | 18 | 0 |
| 8 | [2, 1, 0, 2] | 1 | 12 | 0 |
| 8 | [2, 1, 0, 3] | 1 | 6 | 0 |
| 8 | [2, 1, 1, 0] | 1 | 12 | 0 |
| 8 | [2, 1, 1, 1] | 1 | 6 | 0 |
| 8 | [2, 1, 1, 2] | 1 | 0 | 0 |
| 8 | [3, 0, 0, 0] | 1 | -2 | 2 |
| 8 | [3, 0, 0, 0] | 2 | 18 | 0 |
| 8 | [3, 0, 0, 1] | 1 | -8 | 2 |
| 8 | [3, 0, 0, 1] | 3 | 12 | 0 |
| 8 | [3, 0, 0, 2] | 3 | 6 | 0 |
| 8 | [3, 0, 0, 3] | 3 | 0 | 0 |
| 8 | [3, 0, 0, 4] | 1 | -6 | 0 |
| 8 | [3, 0, 1, 0] | 1 | 6 | 0 |
| 8 | [3, 0, 1, 1] | 1 | 0 | 0 |
| 8 | [3, 0, 1, 2] | 1 | -6 | 0 |
| 8 | [4, 0, 0, 0] | 1 | 24 | 0 |
| 8 | [4, 0, 0, 1] | 1 | 18 | 0 |
| 8 | [4, 0, 0, 2] | 1 | 12 | 0 |
| 8 | [4, 0, 0, 3] | 1 | 6 | 0 |
| 8 | [4, 0, 0, 4] | 1 | 0 | 0 |
| 9 | [0, 0, 0, 1] | 4 | 4 | -1 |
| 9 | [0, 0, 0, 2] | 3 | -2 | -1 |
| 9 | [0, 0, 0, 3] | 2 | -8 | -1 |
| 9 | [0, 0, 0, 3] | 1 | 12 | -3 |
| 9 | [0, 0, 0, 4] | 1 | -14 | -1 |
| 9 | [0, 0, 1, 0] | 4 | -2 | -1 |
| 9 | [0, 0, 1, 1] | 3 | -8 | -1 |
| 9 | [0, 0, 1, 2] | 2 | -14 | -1 |
| 9 | [0, 0, 1, 2] | 1 | 6 | -3 |
| 9 | [0, 0, 1, 3] | 1 | -20 | -1 |
| 9 | [0, 0, 2, 1] | 1 | 0 | -3 |
| 9 | [0, 0, 3, 0] | 1 | -6 | -3 |
| 9 | [0, 1, 0, 0] | 4 | 2 | 1 |
| 9 | [0, 1, 0, 1] | 3 | -4 | 1 |
| 9 | [0, 1, 0, 2] | 2 | -10 | 1 |
| 9 | [0, 1, 0, 2] | 2 | 10 | -1 |
| 9 | [0, 1, 0, 3] | 1 | -16 | 1 |
| 9 | [0, 1, 0, 3] | 1 | 4 | -1 |
| 9 | [0, 1, 1, 1] | 2 | 4 | -1 |
| 9 | [0, 1, 1, 2] | 1 | -2 | -1 |
| 9 | [0, 1, 2, 0] | 2 | -2 | -1 |
| 9 | [0, 1, 2, 1] | 1 | -8 | -1 |
| 9 | [0, 2, 0, 1] | 2 | 8 | 1 |
| 9 | [0, 2, 0, 2] | 1 | 2 | 1 |
| 9 | [0, 2, 1, 0] | 2 | 2 | 1 |
| 9 | [0, 2, 1, 1] | 1 | -4 | 1 |
| 9 | [0, 3, 0, 0] | 1 | 6 | 3 |
| 9 | [1, 0, 0, 0] | 4 | -4 | 1 |
| 9 | [1, 0, 0, 1] | 3 | -10 | 1 |
| 9 | [1, 0, 0, 1] | 3 | 10 | -1 |
| 9 | [1, 0, 0, 2] | 2 | -16 | 1 |
| 9 | [1, 0, 0, 2] | 5 | 4 | -1 |
| 9 | [1, 0, 0, 3] | 1 | -22 | 1 |
| 9 | [1, 0, 0, 3] | 3 | -2 | -1 |
| 9 | [1, 0, 0, 4] | 1 | -8 | -1 |
| 9 | [1, 0, 1, 0] | 3 | 4 | -1 |
| 9 | [1, 0, 1, 1] | 5 | -2 | -1 |
| 9 | [1, 0, 1, 2] | 3 | -8 | -1 |
| 9 | [1, 0, 1, 3] | 1 | -14 | -1 |
| 9 | [1, 0, 2, 0] | 2 | -8 | -1 |
| 9 | [1, 0, 2, 1] | 1 | -14 | -1 |
| 9 | [1, 1, 0, 0] | 3 | 8 | 1 |
| 9 | [1, 1, 0, 1] | 5 | 2 | 1 |
| 9 | [1, 1, 0, 2] | 3 | -4 | 1 |
| 9 | [1, 1, 0, 2] | 1 | 16 | -1 |
| 9 | [1, 1, 0, 3] | 1 | -10 | 1 |
| 9 | [1, 1, 0, 3] | 1 | 10 | -1 |
| 9 | [1, 1, 1, 0] | 2 | -4 | 1 |
| 9 | [1, 1, 1, 1] | 1 | -10 | 1 |
| 9 | [1, 1, 1, 1] | 1 | 10 | -1 |
| 9 | [1, 1, 1, 2] | 1 | 4 | -1 |
| 9 | [1, 1, 2, 0] | 1 | 4 | -1 |
| 9 | [1, 1, 2, 1] | 1 | -2 | -1 |
| 9 | [1, 2, 0, 0] | 1 | 0 | 3 |
| 9 | [1, 2, 0, 1] | 1 | 14 | 1 |
| 9 | [1, 2, 0, 2] | 1 | 8 | 1 |
| 9 | [1, 2, 1, 0] | 1 | 8 | 1 |
| 9 | [1, 2, 1, 1] | 1 | 2 | 1 |
| 9 | [2, 0, 0, 0] | 3 | 2 | 1 |
| 9 | [2, 0, 0, 1] | 5 | -4 | 1 |
| 9 | [2, 0, 0, 1] | 2 | 16 | -1 |
| 9 | [2, 0, 0, 2] | 3 | -10 | 1 |
| 9 | [2, 0, 0, 2] | 3 | 10 | -1 |
| 9 | [2, 0, 0, 3] | 1 | -16 | 1 |
| 9 | [2, 0, 0, 3] | 3 | 4 | -1 |
| 9 | [2, 0, 0, 4] | 1 | -2 | -1 |
| 9 | [2, 0, 1, 0] | 2 | -10 | 1 |
| 9 | [2, 0, 1, 0] | 2 | 10 | -1 |
| 9 | [2, 0, 1, 1] | 1 | -16 | 1 |
| 9 | [2, 0, 1, 1] | 3 | 4 | -1 |
| 9 | [2, 0, 1, 2] | 3 | -2 | -1 |
| 9 | [2, 0, 1, 3] | 1 | -8 | -1 |
| 9 | [2, 0, 2, 0] | 1 | -2 | -1 |
| 9 | [2, 0, 2, 1] | 1 | -8 | -1 |
| 9 | [2, 1, 0, 0] | 1 | -6 | 3 |
| 9 | [2, 1, 0, 0] | 2 | 14 | 1 |
| 9 | [2, 1, 0, 1] | 3 | 8 | 1 |
| 9 | [2, 1, 0, 2] | 3 | 2 | 1 |
| 9 | [2, 1, 0, 3] | 1 | -4 | 1 |
| 9 | [2, 1, 1, 0] | 1 | 2 | 1 |
| 9 | [2, 1, 1, 1] | 1 | -4 | 1 |
| 9 | [3, 0, 0, 0] | 1 | -12 | 3 |
| 9 | [3, 0, 0, 0] | 2 | 8 | 1 |
| 9 | [3, 0, 0, 1] | 3 | 2 | 1 |
| 9 | [3, 0, 0, 1] | 1 | 22 | -1 |
| 9 | [3, 0, 0, 2] | 3 | -4 | 1 |
| 9 | [3, 0, 0, 2] | 1 | 16 | -1 |
| 9 | [3, 0, 0, 3] | 1 | -10 | 1 |
| 9 | [3, 0, 0, 3] | 1 | 10 | -1 |
| 9 | [3, 0, 0, 4] | 1 | 4 | -1 |
| 9 | [3, 0, 1, 0] | 1 | -4 | 1 |
| 9 | [3, 0, 1, 0] | 1 | 16 | -1 |
| 9 | [3, 0, 1, 1] | 1 | -10 | 1 |
| 9 | [3, 0, 1, 1] | 1 | 10 | -1 |
| 9 | [3, 0, 1, 2] | 1 | 4 | -1 |
| 9 | [3, 0, 1, 3] | 1 | -2 | -1 |
| 9 | [3, 1, 0, 0] | 1 | 20 | 1 |
| 9 | [3, 1, 0, 1] | 1 | 14 | 1 |
| 9 | [3, 1, 0, 2] | 1 | 8 | 1 |
| 9 | [3, 1, 0, 3] | 1 | 2 | 1 |
| 9 | [4, 0, 0, 0] | 1 | 14 | 1 |
| 9 | [4, 0, 0, 1] | 1 | 8 | 1 |
| 9 | [4, 0, 0, 2] | 1 | 2 | 1 |
| 9 | [4, 0, 0, 3] | 1 | -4 | 1 |
| 10 | [0, 0, 0, 0] | 6 | 0 | 0 |
| 10 | [0, 0, 0, 1] | 5 | -6 | 0 |
| 10 | [0, 0, 0, 2] | 4 | -12 | 0 |
| 10 | [0, 0, 0, 2] | 3 | 8 | -2 |
| 10 | [0, 0, 0, 3] | 3 | -18 | 0 |
| 10 | [0, 0, 0, 3] | 2 | 2 | -2 |
| 10 | [0, 0, 0, 4] | 2 | -24 | 0 |
| 10 | [0, 0, 0, 4] | 1 | -4 | -2 |
| 10 | [0, 0, 0, 5] | 1 | -30 | 0 |
| 10 | [0, 0, 1, 1] | 3 | 2 | -2 |
| 10 | [0, 0, 1, 2] | 2 | -4 | -2 |
| 10 | [0, 0, 1, 3] | 1 | -10 | -2 |
| 10 | [0, 0, 2, 0] | 3 | -4 | -2 |
| 10 | [0, 0, 2, 1] | 2 | -10 | -2 |
| 10 | [0, 0, 2, 2] | 1 | -16 | -2 |
| 10 | [0, 1, 0, 1] | 4 | 6 | 0 |
| 10 | [0, 1, 0, 2] | 3 | 0 | 0 |
| 10 | [0, 1, 0, 3] | 2 | -6 | 0 |
| 10 | [0, 1, 0, 3] | 1 | 14 | -2 |
| 10 | [0, 1, 0, 4] | 1 | -12 | 0 |
| 10 | [0, 1, 1, 0] | 4 | 0 | 0 |
| 10 | [0, 1, 1, 1] | 3 | -6 | 0 |
| 10 | [0, 1, 1, 2] | 2 | -12 | 0 |
| 10 | [0, 1, 1, 2] | 1 | 8 | -2 |
| 10 | [0, 1, 1, 3] | 1 | -18 | 0 |
| 10 | [0, 1, 2, 1] | 1 | 2 | -2 |
| 10 | [0, 1, 3, 0] | 1 | -4 | -2 |
| 10 | [0, 2, 0, 0] | 3 | 4 | 2 |
| 10 | [0, 2, 0, 1] | 2 | -2 | 2 |
| 10 | [0, 2, 0, 2] | 1 | -8 | 2 |
| 10 | [0, 2, 0, 2] | 2 | 12 | 0 |
| 10 | [0, 2, 0, 3] | 1 | 6 | 0 |
| 10 | [0, 2, 1, 1] | 2 | 6 | 0 |
| 10 | [0, 2, 1, 2] | 1 | 0 | 0 |
| 10 | [0, 2, 2, 0] | 2 | 0 | 0 |
| 10 | [0, 2, 2, 1] | 1 | -6 | 0 |
| 10 | [0, 3, 0, 1] | 1 | 10 | 2 |
| 10 | [0, 3, 1, 0] | 1 | 4 | 2 |
| 10 | [1, 0, 0, 0] | 5 | 6 | 0 |
| 10 | [1, 0, 0, 1] | 9 | 0 | 0 |
| 10 | [1, 0, 0, 2] | 7 | -6 | 0 |
| 10 | [1, 0, 0, 2] | 2 | 14 | -2 |
| 10 | [1, 0, 0, 3] | 5 | -12 | 0 |
| 10 | [1, 0, 0, 3] | 3 | 8 | -2 |
| 10 | [1, 0, 0, 4] | 3 | -18 | 0 |
| 10 | [1, 0, 0, 4] | 1 | 2 | -2 |
| 10 | [1, 0, 0, 5] | 1 | -24 | 0 |
| 10 | [1, 0, 1, 0] | 4 | -6 | 0 |
| 10 | [1, 0, 1, 1] | 3 | -12 | 0 |
| 10 | [1, 0, 1, 1] | 2 | 8 | -2 |
| 10 | [1, 0, 1, 2] | 2 | -18 | 0 |
| 10 | [1, 0, 1, 2] | 3 | 2 | -2 |
| 10 | [1, 0, 1, 3] | 1 | -24 | 0 |
| 10 | [1, 0, 1, 3] | 1 | -4 | -2 |
| 10 | [1, 0, 2, 0] | 2 | 2 | -2 |
| 10 | [1, 0, 2, 1] | 3 | -4 | -2 |
| 10 | [1, 0, 2, 2] | 1 | -10 | -2 |
| 10 | [1, 0, 3, 0] | 1 | -10 | -2 |
| 10 | [1, 1, 0, 0] | 3 | -2 | 2 |
| 10 | [1, 1, 0, 1] | 2 | -8 | 2 |
| 10 | [1, 1, 0, 1] | 3 | 12 | 0 |
| 10 | [1, 1, 0, 2] | 1 | -14 | 2 |
| 10 | [1, 1, 0, 2] | 5 | 6 | 0 |
| 10 | [1, 1, 0, 3] | 3 | 0 | 0 |
| 10 | [1, 1, 0, 4] | 1 | -6 | 0 |
| 10 | [1, 1, 1, 0] | 3 | 6 | 0 |
| 10 | [1, 1, 1, 1] | 5 | 0 | 0 |
| 10 | [1, 1, 1, 2] | 3 | -6 | 0 |
| 10 | [1, 1, 1, 3] | 1 | -12 | 0 |
| 10 | [1, 1, 2, 0] | 2 | -6 | 0 |
| 10 | [1, 1, 2, 1] | 1 | -12 | 0 |
| 10 | [1, 2, 0, 0] | 2 | 10 | 2 |
| 10 | [1, 2, 0, 1] | 3 | 4 | 2 |
| 10 | [1, 2, 0, 2] | 1 | -2 | 2 |
| 10 | [1, 2, 0, 2] | 1 | 18 | 0 |
| 10 | [1, 2, 0, 3] | 1 | 12 | 0 |
| 10 | [1, 2, 1, 0] | 1 | -2 | 2 |
| 10 | [1, 2, 1, 1] | 1 | 12 | 0 |
| 10 | [1, 2, 1, 2] | 1 | 6 | 0 |
| 10 | [1, 2, 2, 0] | 1 | 6 | 0 |
| 10 | [1, 2, 2, 1] | 1 | 0 | 0 |
| 10 | [2, 0, 0, 0] | 3 | -8 | 2 |
| 10 | [2, 0, 0, 0] | 4 | 12 | 0 |
| 10 | [2, 0, 0, 1] | 2 | -14 | 2 |
| 10 | [2, 0, 0, 1] | 7 | 6 | 0 |
| 10 | [2, 0, 0, 2] | 1 | -20 | 2 |
| 10 | [2, 0, 0, 2] | 9 | 0 | 0 |
| 10 | [2, 0, 0, 2] | 1 | 20 | -2 |
| 10 | [2, 0, 0, 3] | 6 | -6 | 0 |
| 10 | [2, 0, 0, 3] | 1 | 14 | -2 |
| 10 | [2, 0, 0, 4] | 3 | -12 | 0 |
| 10 | [2, 0, 0, 4] | 1 | 8 | -2 |
| 10 | [2, 0, 0, 5] | 1 | -18 | 0 |
| 10 | [2, 0, 1, 0] | 3 | 0 | 0 |
| 10 | [2, 0, 1, 1] | 5 | -6 | 0 |
| 10 | [2, 0, 1, 1] | 1 | 14 | -2 |
| 10 | [2, 0, 1, 2] | 3 | -12 | 0 |
| 10 | [2, 0, 1, 2] | 1 | 8 | -2 |
| 10 | [2, 0, 1, 3] | 1 | -18 | 0 |
| 10 | [2, 0, 1, 3] | 1 | 2 | -2 |
| 10 | [2, 0, 2, 0] | 2 | -12 | 0 |
| 10 | [2, 0, 2, 0] | 1 | 8 | -2 |
| 10 | [2, 0, 2, 1] | 1 | -18 | 0 |
| 10 | [2, 0, 2, 1] | 1 | 2 | -2 |
| 10 | [2, 0, 2, 2] | 1 | -4 | -2 |
| 10 | [2, 1, 0, 0] | 2 | 4 | 2 |
| 10 | [2, 1, 0, 1] | 3 | -2 | 2 |
| 10 | [2, 1, 0, 1] | 2 | 18 | 0 |
| 10 | [2, 1, 0, 2] | 1 | -8 | 2 |
| 10 | [2, 1, 0, 2] | 3 | 12 | 0 |
| 10 | [2, 1, 0, 3] | 3 | 6 | 0 |
| 10 | [2, 1, 0, 4] | 1 | 0 | 0 |
| 10 | [2, 1, 1, 0] | 1 | -8 | 2 |
| 10 | [2, 1, 1, 0] | 2 | 12 | 0 |
| 10 | [2, 1, 1, 1] | 3 | 6 | 0 |
| 10 | [2, 1, 1, 2] | 3 | 0 | 0 |
| 10 | [2, 1, 1, 3] | 1 | -6 | 0 |
| 10 | [2, 1, 2, 0] | 1 | 0 | 0 |
| 10 | [2, 1, 2, 1] | 1 | -6 | 0 |
| 10 | [2, 2, 0, 0] | 1 | 16 | 2 |
| 10 | [2, 2, 0, 1] | 1 | 10 | 2 |
| 10 | [2, 2, 0, 2] | 1 | 4 | 2 |
| 10 | [3, 0, 0, 0] | 2 | -2 | 2 |
| 10 | [3, 0, 0, 0] | 3 | 18 | 0 |
| 10 | [3, 0, 0, 1] | 3 | -8 | 2 |
| 10 | [3, 0, 0, 1] | 5 | 12 | 0 |
| 10 | [3, 0, 0, 2] | 1 | -14 | 2 |
| 10 | [3, 0, 0, 2] | 6 | 6 | 0 |
| 10 | [3, 0, 0, 3] | 6 | 0 | 0 |
| 10 | [3, 0, 0, 4] | 3 | -6 | 0 |
| 10 | [3, 0, 0, 5] | 1 | -12 | 0 |
| 10 | [3, 0, 1, 0] | 1 | -14 | 2 |
| 10 | [3, 0, 1, 0] | 2 | 6 | 0 |
| 10 | [3, 0, 1, 1] | 3 | 0 | 0 |
| 10 | [3, 0, 1, 2] | 3 | -6 | 0 |
| 10 | [3, 0, 1, 3] | 1 | -12 | 0 |
| 10 | [3, 0, 2, 0] | 1 | -6 | 0 |
| 10 | [3, 0, 2, 1] | 1 | -12 | 0 |
| 10 | [3, 1, 0, 0] | 1 | 10 | 2 |
| 10 | [3, 1, 0, 1] | 1 | 4 | 2 |
| 10 | [3, 1, 0, 1] | 1 | 24 | 0 |
| 10 | [3, 1, 0, 2] | 1 | -2 | 2 |
| 10 | [3, 1, 0, 2] | 1 | 18 | 0 |
| 10 | [3, 1, 0, 3] | 1 | 12 | 0 |
| 10 | [3, 1, 0, 4] | 1 | 6 | 0 |
| 10 | [3, 1, 1, 0] | 1 | 18 | 0 |
| 10 | [3, 1, 1, 1] | 1 | 12 | 0 |
| 10 | [3, 1, 1, 2] | 1 | 6 | 0 |
| 10 | [3, 1, 1, 3] | 1 | 0 | 0 |
| 10 | [4, 0, 0, 0] | 1 | 4 | 2 |
| 10 | [4, 0, 0, 0] | 2 | 24 | 0 |
| 10 | [4, 0, 0, 1] | 1 | -2 | 2 |
| 10 | [4, 0, 0, 1] | 3 | 18 | 0 |
| 10 | [4, 0, 0, 2] | 1 | -8 | 2 |
| 10 | [4, 0, 0, 2] | 3 | 12 | 0 |
| 10 | [4, 0, 0, 3] | 3 | 6 | 0 |
| 10 | [4, 0, 0, 4] | 3 | 0 | 0 |
| 10 | [4, 0, 0, 5] | 1 | -6 | 0 |
| 10 | [4, 0, 1, 0] | 1 | 12 | 0 |
| 10 | [4, 0, 1, 1] | 1 | 6 | 0 |
| 10 | [4, 0, 1, 2] | 1 | 0 | 0 |
| 10 | [4, 0, 1, 3] | 1 | -6 | 0 |
| 10 | [5, 0, 0, 0] | 1 | 30 | 0 |
| 10 | [5, 0, 0, 1] | 1 | 24 | 0 |
| 10 | [5, 0, 0, 2] | 1 | 18 | 0 |
| 10 | [5, 0, 0, 3] | 1 | 12 | 0 |
| 10 | [5, 0, 0, 4] | 1 | 6 | 0 |
| 10 | [5, 0, 0, 5] | 1 | 0 | 0 |

</details>

Hand checks: $35\to24_0+1_0+5_{+6}+\bar5_{-6}$; $15\to10_{+2}+5_{-4}$; $\bar{15}\to\bar{10}_{-2}+\bar5_{+4}$.

**Degree 2:** $2\,1_{(0,0)}+24_{(0,0)}+5_{(6,0)}+\bar5_{(-6,0)}$. **Degree 3:** $10_{(2,1)}+5_{(-4,1)}+\bar{10}_{(-2,-1)}+\bar5_{(4,-1)}$.

### Complete branched candidate generators
| degree | representation | mult. | dimension | x | q | source/classification |
|---|---|---|---|---|---|---|
| 2 | [0, 0, 0, 0] | 2 | 1 | 0 | 0 | stored result |
| 2 | [0, 0, 0, 1] | 1 | 5 | -6 | 0 | stored result |
| 2 | [1, 0, 0, 0] | 1 | 5 | 6 | 0 | stored result |
| 2 | [1, 0, 0, 1] | 1 | 24 | 0 | 0 | stored result |
| 3 | [0, 0, 0, 1] | 1 | 5 | 4 | -1 | stored result |
| 3 | [0, 0, 1, 0] | 1 | 10 | -2 | -1 | stored result |
| 3 | [0, 1, 0, 0] | 1 | 10 | 2 | 1 | stored result |
| 3 | [1, 0, 0, 0] | 1 | 5 | -4 | 1 | stored result |

### Complete branched first relations
| degree | representation | mult. | dimension | x | q | source/classification |
|---|---|---|---|---|---|---|
| 4 | [0, 0, 0, 0] | -2 | 1 | 0 | 0 | stored result |
| 4 | [0, 0, 0, 1] | -1 | 5 | -6 | 0 | stored result |
| 4 | [1, 0, 0, 0] | -1 | 5 | 6 | 0 | stored result |
| 4 | [1, 0, 0, 1] | -1 | 24 | 0 | 0 | stored result |

## 11. How checks are organised

PASS is assigned only to explicit successful machine evidence. Unknown structures are UNAVAILABLE. Empty reconstruction differences count only alongside the explicit equality schema/check.

| notebook ID | source key | stage | claim | type | validation target | actual | status | evidence | commit | re-evaluated |
|---|---|---|---|---|---|---|---|---|---|---|
| NB-HWG-ALL-MULTIPLICITIES-ARE-INTEGERS-A660DE | validation_results.all_multiplicities_are_integers | hwg | all multiplicities are integers | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-ALL-MULTIPLICITIES-ARE-NONNEGATIVE-998368 | validation_results.all_multiplicities_are_nonnegative | hwg | all multiplicities are nonnegative | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-ALL-PASSED-149BD5 | validation_results.all_passed | hwg | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-CONSTANT-COEFFICIENT-IS-ONE-7EDAD8 | validation_results.constant_coefficient_is_one | hwg | constant coefficient is one | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-NO-TERMS-ABOVE-REQUESTED-DEGREE-652FED | validation_results.no_terms_above_requested_degree | hwg | no terms above requested degree | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-PE-EQUALS-RATIONAL-PRODUCT-0FDDCA | validation_results.pe_equals_rational_product | hwg | pe equals rational product | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-TRUNCATION-STABILITY-0E4777 | validation_results.truncation_stability | hwg | truncation stability | regression | stored expected invariant identified by the producing stage | {"0": true, "10": true, "2": true, "4": true, "6": true, "8": true} | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-ALL-MULTIPLICITIES-ARE-INTEGERS-A660DE | validation_results.all_multiplicities_are_integers | hwg | all multiplicities are integers | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-ALL-MULTIPLICITIES-ARE-NONNEGATIVE-998368 | validation_results.all_multiplicities_are_nonnegative | hwg | all multiplicities are nonnegative | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-ALL-PASSED-149BD5 | validation_results.all_passed | hwg | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-CONSTANT-COEFFICIENT-IS-ONE-7EDAD8 | validation_results.constant_coefficient_is_one | hwg | constant coefficient is one | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-NO-TERMS-ABOVE-REQUESTED-DEGREE-652FED | validation_results.no_terms_above_requested_degree | hwg | no terms above requested degree | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-PE-EQUALS-RATIONAL-PRODUCT-0FDDCA | validation_results.pe_equals_rational_product | hwg | pe equals rational product | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-HWG-TRUNCATION-STABILITY-0E4777 | validation_results.truncation_stability | hwg | truncation stability | regression | stored expected invariant identified by the producing stage | {"0": true, "10": true, "2": true, "4": true, "6": true, "8": true} | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/checks.json | 3d3a7bae82c586b7363ada08221758bb90f4f168 | no |
| NB-CHAR-ALL-PASSED-7B5F57 | validation_results.all_passed | characters | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-DIMENSIONS-NONNEGATIVE-INTEGERS-03E5B0 | validation_results.dimensions_nonnegative_integers | characters | dimensions nonnegative integers | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-LEADING-COEFFICIENTS-B23D64 | validation_results.leading_coefficients | characters | leading coefficients | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-ONE-IRREP-PER-HWG-MONOMIAL-BEFORE-COMBINING-BF4BB2 | validation_results.one_irrep_per_hwg_monomial_before_combining | characters | one irrep per hwg monomial before combining | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-Q-CHARGES-PRESERVED-194BCD | validation_results.q_charges_preserved | characters | q charges preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-Q-EQUALS-ONE-MATCHES-UNREFINED-357DAC | validation_results.q_equals_one_matches_unrefined | characters | q equals one matches unrefined | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-REPRESENTATION-MULTIPLICITIES-NONNEGATIVE-INTEGERS-2693EB | validation_results.representation_multiplicities_nonnegative_integers | characters | representation multiplicities nonnegative integers | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-CHAR-T-DEGREES-PRESERVED-6C26A3 | validation_results.t_degrees_preserved | characters | t degrees preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json | d562e55b97222ee53371097c7a3b1818f5550a22 | no |
| NB-PL-ALL-FINAL-COEFFICIENTS-INTEGRAL-EFBA6F | validation_results.all_final_coefficients_integral | plethystic-log | all final coefficients integral | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-ALL-PASSED-5FA8BC | validation_results.all_passed | plethystic-log | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-DEGREE-2-INDEPENDENT-VALUE-9F9007 | validation_results.degree_2_independent_value | plethystic-log | degree 2 independent value | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-DEGREE-3-INDEPENDENT-VALUE-2B1E38 | validation_results.degree_3_independent_value | plethystic-log | degree 3 independent value | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-DEGREE-4-INDEPENDENT-VALUE-24017E | validation_results.degree_4_independent_value | plethystic-log | degree 4 independent value | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-DEGREES-TRUNCATED-E37BE2 | validation_results.degrees_truncated | plethystic-log | degrees truncated | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-DIRECT-SCALAR-MATCHES-REFINED-UNREFINEMENT-18C4A8 | validation_results.direct_scalar_matches_refined_unrefinement | plethystic-log | direct scalar matches refined unrefinement | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-PL-NEGATIVE-COEFFICIENTS-RETAINED-8CE499 | validation_results.negative_coefficients_retained | plethystic-log | negative coefficients retained | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json | ebc32f70bed45f167dd50eb67152a646d3c0185d | no |
| NB-RECON-ALL-PASSED-090256 | validation_results.all_passed | reconstruction | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-DIFFERENCE-IS-EMPTY-D41C49 | validation_results.difference_is_empty | reconstruction | difference is empty | round-trip reconstruction | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-INDEPENDENT-SCALAR-PE-EQUAL-89BBC1 | validation_results.independent_scalar_pe_equal | reconstruction | independent scalar pe equal | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-MULTIPLICITIES-INTEGRAL-319D5D | validation_results.multiplicities_integral | reconstruction | multiplicities integral | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-MULTIPLICITIES-NONNEGATIVE-FEE6C7 | validation_results.multiplicities_nonnegative | reconstruction | multiplicities nonnegative | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-ORDER-STABILITY-58D959 | validation_results.order_stability | reconstruction | order stability | regression | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-Q-REFINED-DIMENSIONS-EQUAL-F01E6D | validation_results.q_refined_dimensions_equal | reconstruction | q refined dimensions equal | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-REFINED-CHARACTER-SERIES-EQUAL-028BF2 | validation_results.refined_character_series_equal | reconstruction | refined character series equal | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-RECON-UNREFINED-DIMENSIONS-EQUAL-92E74F | validation_results.unrefined_dimensions_equal | reconstruction | unrefined dimensions equal | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json | d55b1caf1e54e50083ec2a330f588913788e0104 | no |
| NB-OPER-ALL-PASSED-7F0314 | validation_results.all_passed | operator-analysis | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json | 7c50c0d7fa9287df82aed756d311f81dcb61c343 | no |
| NB-OPER-DEFICIT-EQUALS-NEGATIVE-PL-CONTENT-5087F8 | validation_results.deficit_equals_negative_pl_content | operator-analysis | deficit equals negative pl content | independent-route comparison | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json | 7c50c0d7fa9287df82aed756d311f81dcb61c343 | no |
| NB-OPER-DIMENSIONS-EXACT-FD7A9A | validation_results.dimensions_exact | operator-analysis | dimensions exact | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json | 7c50c0d7fa9287df82aed756d311f81dcb61c343 | no |
| NB-OPER-FIRST-NEGATIVE-DEGREE-EXISTS-14B9F4 | validation_results.first_negative_degree_exists | operator-analysis | first negative degree exists | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json | 7c50c0d7fa9287df82aed756d311f81dcb61c343 | no |
| NB-OPER-RELATION-DEGREE-NOT-MIXED-17C876 | validation_results.relation_degree_not_mixed | operator-analysis | relation degree not mixed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json | 7c50c0d7fa9287df82aed756d311f81dcb61c343 | no |
| NB-BRANCH-ALL-PARENT-DIMENSIONS-PRESERVED-A7F1EB | validation_results.all_parent_dimensions_preserved | branching | all parent dimensions preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-ALL-PASSED-A49E09 | validation_results.all_passed | branching | all passed | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-CANDIDATE-GENERATOR-DIMENSION-82D7F6 | validation_results.candidate_generator_dimension | branching | candidate generator dimension | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-CHARACTER-UNREFINEMENT-PRESERVED-750FA6 | validation_results.character_unrefinement_preserved | branching | character unrefinement preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-FIRST-RELATION-DIMENSION-05AA5A | validation_results.first_relation_dimension | branching | first relation dimension | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-ORDINARY-MULTIPLICITIES-NONNEGATIVE-INTEGERS-C445D1 | validation_results.ordinary_multiplicities_nonnegative_integers | branching | ordinary multiplicities nonnegative integers | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-PHYSICAL-CHARGE-MAP-ASSUMED-82B2DF | validation_results.physical_charge_map_assumed | branching | physical charge map assumed | property test | stored expected invariant identified by the producing stage | false | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-PLETHYSTIC-LOG-UNREFINEMENT-PRESERVED-9109E6 | validation_results.plethystic_log_unrefinement_preserved | branching | plethystic log unrefinement preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-Q-CHARGES-PRESERVED-04AB6C | validation_results.q_charges_preserved | branching | q charges preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-RAW-CHARGE-BASIS-ONLY-ADDAB0 | validation_results.raw_charge_basis_only | branching | raw charge basis only | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-RECONSTRUCTED-CHARACTER-BRANCHING-EQUAL-A30CE6 | validation_results.reconstructed_character_branching_equal | branching | reconstructed character branching equal | round-trip reconstruction | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-SIGNED-VIRTUAL-MULTIPLICITIES-RETAINED-06925D | validation_results.signed_virtual_multiplicities_retained | branching | signed virtual multiplicities retained | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-T-DEGREES-PRESERVED-EF8BCC | validation_results.t_degrees_preserved | branching | t degrees preserved | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-X-CHARGES-ARE-EXACT-INTEGERS-E454CE | validation_results.x_charges_are_exact_integers | branching | x charges are exact integers | property test | stored expected invariant identified by the producing stage | true | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-3-0-66CB7D | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 5040 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-2-1-BB8D7C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2205 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-2-2-8D284F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 18480 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-2-0-F537FF | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 6720 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-2-0-2-1-2032EB | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 93555 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 405 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-1-2-73AF97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 29700 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-0-0-3-DE5AA8 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2695 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-1-0-1-3-9DB37C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 154791 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-4-0-0-0-4-1B1AC7 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 12740 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-5-0-0-0-5-D2F865 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 47628 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-3-0-1-0-D2C728 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 5040 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-2-0-0-1-66D755 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2205 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-2-0-0-2-F4B444 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 18480 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 405 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 405 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-0-0-3-DE5AA8 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2695 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-2-0-7B9D8A | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1176 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-1-2-1744B7 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-1-0-AA1767 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1176 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-0-2-F2D8C0 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-2-1-BB8D7C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2205 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-2-0-F537FF | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 6720 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 405 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-1-2-73AF97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 29700 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-0-0-3-DE5AA8 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2695 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-4-0-0-0-4-1B1AC7 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 12740 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-2-0-0-1-66D755 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2205 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-3-0-BFC810 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 490 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-2-0-7B9D8A | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1176 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-2-1-40E138 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 19200 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-1-2-1744B7 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-0-1-3-F1B4B4 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21504 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-1-0-AA1767 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1176 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-2-0-1-1-1115E7 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 19200 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-0-2-F2D8C0 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-1-0-0-3-76EAB4 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21504 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-3-0-0-0-CD61F4 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 490 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-2-97E2BE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 210 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-2-9EED6D | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1701 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-1-0-28C4F9 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1050 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-1-0-47C4FB | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2430 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-0-0-1-2750BE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 315 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-2-2-628F6E | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1134 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-3-0-BFC810 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 490 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-0-3-8AEB48 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 840 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-1-1-78228C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 896 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-2-0-0-01199B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 175 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-2-20C623 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 280 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-3-0-0-0-CD61F4 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 490 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-2-0-1-45357C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3969 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 3675 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-1-0-0-B4EEF2 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 896 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 405 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-1-0-B41EA8 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 280 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-2-0-0-0-C2339F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1134 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-1-0-0-D8BC3C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 840 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-2-001385 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 2430 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-1-0-1-F82C0F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1050 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-3-C49459 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 315 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-1-0-1-51D7B4 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1701 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-0-0-BA2108 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 210 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-2-0-0-01199B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 175 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-0-0-BA2108 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 210 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-2-97E2BE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 210 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-1-1-78228C | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 896 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-2-0-0-01199B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 175 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-2-20C623 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 280 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 189 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-1-0-0-B4EEF2 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 896 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-1-0-B41EA8 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 280 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-2-1-0-F892A5 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1470 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-1-0-1-F82C0F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1050 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-3-C49459 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 315 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-1-0-1-51D7B4 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1701 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-1-0-0-0-BA2108 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 210 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-4-0-0-0-0-A4900A | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 126 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-4-CAC6D9 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 126 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-2-97E2BE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 210 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 105 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-2-0-0-D4866F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1470 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-0-2-9EED6D | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1701 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-1-1-0-28C4F9 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1050 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 384 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 21 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-3-0-0-0-1-2750BE | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 315 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 15 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 1 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | dimension_checks | branching | parent and child dimensions agree | property test | sum of child dimensions equals parent Weyl dimension | 35 | ✅ **PASS** | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json | a49d1b010160ba3d31f70c2cdba5a6fe658bf572 | no |

**Totals by status:** PASS=242, FAIL=0, PENDING=0, UNAVAILABLE=0, NOT APPLICABLE=0. **By stage:** branching=198, characters=8, hwg=14, operator-analysis=5, plethystic-log=8, reconstruction=9.

## 12. Independent validation benchmarks

This section uses stored evidence only; neither benchmark is rerun. For $SU(3)_{\pm1/2}$ with 9 flavours, stored D10 convention, refined/scalar reconstruction, determinism and input-integrity evidence are distinguished from **literature agreement reported by the user** (external, not machine-executed). For $SU(4)_{\pm1/2}$ with 11 flavours, the stored D12 character Hilbert series through $t^8$ completed; literature agreement is user-reported, while the refined PL was resource-blocked specifically at refined PL computation—not a Hilbert-series failure.

## 13. Implementation guide

|Operation|Module / public API|Data|Tests/evidence|
|---|---|---|---|
|Sparse PE|`expansion.expand_pe`|`SparseSeries`|`tests/test_expansion.py`, `hwg_expansion.json`|
|Restore characters|`characters.restore_characters`|character series|`tests/test_characters.py`|
|Formal log + Möbius PL|`plethystic.plethystic_logarithm`|virtual character series|`tests/test_plethystic.py`|
|Formal exp + PE|`plethystic.plethystic_exponential`|virtual character series|reconstruction checks|
|Branch|`branching.branch_character`|restricted weight dictionaries|`tests/test_branching.py`|

```text
PE: for factor, multiply truncated sparse geometric/binomial series
restore: for monomial, construct WCR irrep at its Dynkin labels
log: repeatedly multiply X=H-1 and add (-1)^(n+1)X^n/n
PL: sum mu(k)/k * log(Adams_k(H))
exp: sum F^n/n!; PE: exp(sum Adams_k(F)/k)
branch: restrict weights → group by x → subtract child highest characters
```

## 14. How to run a new theory

Review each generated JSON/check file before proceeding. Commands use repository Sage Python only.

```bash
# 1–3 add reference, theories/THEORY_ID.yaml, then audit input
./scripts/sage-python -m hwg_pipeline expand THEORY_ID --order ORDER
./scripts/sage-python -m hwg_pipeline characters THEORY_ID --order ORDER
./scripts/sage-python -m hwg_pipeline plethystic-log THEORY_ID --order ORDER
./scripts/sage-python -m hwg_pipeline reconstruct THEORY_ID --order ORDER
./scripts/sage-python -m hwg_pipeline analyze-pl THEORY_ID --order ORDER
./scripts/sage-python -m hwg_pipeline branch THEORY_ID --order ORDER --branching BRANCHING_ID
./scripts/sage-python -m hwg_pipeline latex-report THEORY_ID --order ORDER --branching BRANCHING_ID --through branching --strict
./scripts/sage-python -m hwg_pipeline project-notebook THEORY_ID --order ORDER --branching BRANCHING_ID --through branching --strict
```

## 15. Limitations and next steps

This notebook reaches manifest branching. It does not infer a physical baryon/instanton map, microscopic distinction of neutral singlets, explicit polynomial variables or relations, coordinate-ring ideals, Gröbner bases, or a monopole-formula derivation. Stored charge-map results, where present, are optional and outside the required scope.

# Appendices

## Appendix A. Complete source fixture

```yaml
abelian_factors:
- display_name: U(1)
  fugacity: q
  id: q
chern_simons_convention: absolute value
chern_simons_level: 3/2
coupling: infinite
gauge_algebra: A2
gauge_display_name: SU(3)
grading_variable: t
id: su3_nf5_k3o2_infinite
nonphysical: false
number_of_flavours: 5
pe:
  original_pe_latex: '\PE\!\left[(\mu_1\mu_5+1)t^2+(q\mu_2+q^{-1}\mu_4)t^3

    +\mu_2\mu_4t^4-\mu_2\mu_4t^6\right]'
  terms:
  - coefficient: 1
    monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 1
        - 0
        - 0
        - 0
        - 1
      t_degree: 2
  - coefficient: 1
    monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 0
        - 0
        - 0
        - 0
        - 0
      t_degree: 2
  - coefficient: 1
    monomial:
      abelian_charges:
        q: 1
      representations:
        enhanced:
        - 0
        - 1
        - 0
        - 0
        - 0
      t_degree: 3
  - coefficient: 1
    monomial:
      abelian_charges:
        q: -1
      representations:
        enhanced:
        - 0
        - 0
        - 0
        - 1
        - 0
      t_degree: 3
  - coefficient: 1
    monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 0
        - 1
        - 0
        - 1
        - 0
      t_degree: 4
  - coefficient: -1
    monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 0
        - 1
        - 0
        - 1
        - 0
      t_degree: 6
rational_product:
  factors:
  - monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 0
        - 1
        - 0
        - 1
        - 0
      t_degree: 6
    power: 1
  - monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 0
        - 0
        - 0
        - 0
        - 0
      t_degree: 2
    power: -1
  - monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 1
        - 0
        - 0
        - 0
        - 1
      t_degree: 2
    power: -1
  - monomial:
      abelian_charges:
        q: 1
      representations:
        enhanced:
        - 0
        - 1
        - 0
        - 0
        - 0
      t_degree: 3
    power: -1
  - monomial:
      abelian_charges:
        q: -1
      representations:
        enhanced:
        - 0
        - 0
        - 0
        - 1
        - 0
      t_degree: 3
    power: -1
  - monomial:
      abelian_charges:
        q: 0
      representations:
        enhanced:
        - 0
        - 1
        - 0
        - 1
        - 0
      t_degree: 4
    power: -1
  original_rational_product_latex: '\frac{1-\mu_2\mu_4t^6}

    {(1-t^2)(1-\mu_1\mu_5t^2)(1-q\mu_2t^3)(1-q^{-1}\mu_4t^3)(1-\mu_2\mu_4t^4)}'
simple_factors:
- cartan_type: A
  display_name: SU(6)
  highest_weight_fugacities:
  - mu_1
  - mu_2
  - mu_3
  - mu_4
  - mu_5
  id: enhanced
  rank: 5
source_references:
- description: Committed LaTeX source for the infinite-coupling HWG
  equation: '11.3'
  path: references/overleaf/su3_5f_6f_hwg_results.tex
title: SU(3)+5F at infinite coupling with |k|=3/2
```

## Appendix B. Complete HWG expansion

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 3 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 3 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 4 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 4 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 4 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 4 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 5 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 5 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 5 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 5 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 6 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 6 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 6 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 6 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 6 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 6 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 6 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 6 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 7 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 7 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 7 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 7 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 7 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 7 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 8 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 8 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 8 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 8 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 8 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 8 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 8 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 8 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 8 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 8 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 8 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 9 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,0,0,3,0]_{A5} | 1 | 0 | -3 |
| 9 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 9 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 9 | [0,3,0,0,0]_{A5} | 1 | 0 | 3 |
| 9 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 9 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 9 | [1,1,0,2,1]_{A5} | 1 | 0 | -1 |
| 9 | [1,2,0,1,1]_{A5} | 1 | 0 | 1 |
| 9 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 9 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 9 | [3,0,0,1,3]_{A5} | 1 | 0 | -1 |
| 9 | [3,1,0,0,3]_{A5} | 1 | 0 | 1 |
| 10 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 10 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,1,0,3,0]_{A5} | 1 | 0 | -2 |
| 10 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 10 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,3,0,1,0]_{A5} | 1 | 0 | 2 |
| 10 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 10 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 10 | [1,2,0,2,1]_{A5} | 1 | 0 | 0 |
| 10 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 10 | [2,0,0,2,2]_{A5} | 1 | 0 | -2 |
| 10 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 10 | [2,2,0,0,2]_{A5} | 1 | 0 | 2 |
| 10 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 10 | [3,1,0,1,3]_{A5} | 1 | 0 | 0 |
| 10 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 10 | [5,0,0,0,5]_{A5} | 1 | 0 | 0 |

## Appendix C. Complete character-valued Hilbert series

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 3 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 3 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 4 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 4 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 4 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 4 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 5 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 5 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 5 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 5 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 6 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 6 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 6 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 6 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 6 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 6 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 6 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 6 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 7 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 7 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 7 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 7 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 7 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 7 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 7 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 8 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 8 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 8 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 8 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 8 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 8 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 8 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 8 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 8 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 8 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 8 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 9 | [0,0,0,3,0]_{A5} | 1 | 0 | -3 |
| 9 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,1,0,2,0]_{A5} | 1 | 0 | -1 |
| 9 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 9 | [1,1,0,2,1]_{A5} | 1 | 0 | -1 |
| 9 | [2,0,0,1,2]_{A5} | 1 | 0 | -1 |
| 9 | [3,0,0,1,3]_{A5} | 1 | 0 | -1 |
| 9 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 9 | [0,2,0,1,0]_{A5} | 1 | 0 | 1 |
| 9 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 9 | [1,2,0,1,1]_{A5} | 1 | 0 | 1 |
| 9 | [2,1,0,0,2]_{A5} | 1 | 0 | 1 |
| 9 | [3,1,0,0,3]_{A5} | 1 | 0 | 1 |
| 9 | [0,3,0,0,0]_{A5} | 1 | 0 | 3 |
| 10 | [0,0,0,2,0]_{A5} | 1 | 0 | -2 |
| 10 | [0,1,0,3,0]_{A5} | 1 | 0 | -2 |
| 10 | [1,0,0,2,1]_{A5} | 1 | 0 | -2 |
| 10 | [2,0,0,2,2]_{A5} | 1 | 0 | -2 |
| 10 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,1,0,1,0]_{A5} | 1 | 0 | 0 |
| 10 | [0,2,0,2,0]_{A5} | 1 | 0 | 0 |
| 10 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,1,0,1,1]_{A5} | 1 | 0 | 0 |
| 10 | [1,2,0,2,1]_{A5} | 1 | 0 | 0 |
| 10 | [2,0,0,0,2]_{A5} | 1 | 0 | 0 |
| 10 | [2,1,0,1,2]_{A5} | 1 | 0 | 0 |
| 10 | [3,0,0,0,3]_{A5} | 1 | 0 | 0 |
| 10 | [3,1,0,1,3]_{A5} | 1 | 0 | 0 |
| 10 | [4,0,0,0,4]_{A5} | 1 | 0 | 0 |
| 10 | [5,0,0,0,5]_{A5} | 1 | 0 | 0 |
| 10 | [0,2,0,0,0]_{A5} | 1 | 0 | 2 |
| 10 | [0,3,0,1,0]_{A5} | 1 | 0 | 2 |
| 10 | [1,2,0,0,1]_{A5} | 1 | 0 | 2 |
| 10 | [2,2,0,0,2]_{A5} | 1 | 0 | 2 |

## Appendix D. Complete q-refined and unrefined series

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | — | 1 | 0 | 0 |
| 2 | — | 36 | 0 | 0 |
| 3 | — | 15 | 0 | -1 |
| 3 | — | 15 | 0 | 1 |
| 4 | — | 630 | 0 | 0 |
| 5 | — | 399 | 0 | -1 |
| 5 | — | 399 | 0 | 1 |
| 6 | — | 105 | 0 | -2 |
| 6 | — | 7000 | 0 | 0 |
| 6 | — | 105 | 0 | 2 |
| 7 | — | 5250 | 0 | -1 |
| 7 | — | 5250 | 0 | 1 |
| 8 | — | 2310 | 0 | -2 |
| 8 | — | 56160 | 0 | 0 |
| 8 | — | 2310 | 0 | 2 |
| 9 | — | 490 | 0 | -3 |
| 9 | — | 45954 | 0 | -1 |
| 9 | — | 45954 | 0 | 1 |
| 9 | — | 490 | 0 | 3 |
| 10 | — | 25830 | 0 | -2 |
| 10 | — | 352134 | 0 | 0 |
| 10 | — | 25830 | 0 | 2 |

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | — | 1 | 0 | 0 |
| 1 | — | 0 | 0 | 0 |
| 2 | — | 36 | 0 | 0 |
| 3 | — | 30 | 0 | 0 |
| 4 | — | 630 | 0 | 0 |
| 5 | — | 798 | 0 | 0 |
| 6 | — | 7210 | 0 | 0 |
| 7 | — | 10500 | 0 | 0 |
| 8 | — | 60780 | 0 | 0 |
| 9 | — | 92888 | 0 | 0 |
| 10 | — | 403794 | 0 | 0 |

## Appendix E. Complete refined PL

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 2 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 2 | [1,0,0,0,1]_{A5} | 1 | 0 | 0 |
| 3 | [0,0,0,1,0]_{A5} | 1 | 0 | -1 |
| 3 | [0,1,0,0,0]_{A5} | 1 | 0 | 1 |
| 4 | [0,0,0,0,0]_{A5} | -1 | 0 | 0 |
| 4 | [1,0,0,0,1]_{A5} | -1 | 0 | 0 |
| 5 | [0,0,0,0,2]_{A5} | -1 | 0 | -1 |
| 5 | [0,0,0,1,0]_{A5} | -1 | 0 | -1 |
| 5 | [1,0,1,0,0]_{A5} | -1 | 0 | -1 |
| 5 | [0,0,1,0,1]_{A5} | -1 | 0 | 1 |
| 5 | [0,1,0,0,0]_{A5} | -1 | 0 | 1 |
| 5 | [2,0,0,0,0]_{A5} | -1 | 0 | 1 |
| 6 | [0,1,0,0,0]_{A5} | -1 | 0 | -2 |
| 6 | [0,0,0,0,0]_{A5} | -1 | 0 | 0 |
| 6 | [0,0,2,0,0]_{A5} | -1 | 0 | 0 |
| 6 | [0,1,0,1,0]_{A5} | -1 | 0 | 0 |
| 6 | [0,0,0,1,0]_{A5} | -1 | 0 | 2 |
| 7 | [0,0,0,0,2]_{A5} | 2 | 0 | -1 |
| 7 | [0,0,0,1,0]_{A5} | 2 | 0 | -1 |
| 7 | [1,0,0,1,1]_{A5} | 1 | 0 | -1 |
| 7 | [1,0,1,0,0]_{A5} | 2 | 0 | -1 |
| 7 | [2,1,0,0,0]_{A5} | 1 | 0 | -1 |
| 7 | [0,0,0,1,2]_{A5} | 1 | 0 | 1 |
| 7 | [0,0,1,0,1]_{A5} | 2 | 0 | 1 |
| 7 | [0,1,0,0,0]_{A5} | 2 | 0 | 1 |
| 7 | [1,1,0,0,1]_{A5} | 1 | 0 | 1 |
| 7 | [2,0,0,0,0]_{A5} | 2 | 0 | 1 |
| 8 | [0,0,1,0,1]_{A5} | 2 | 0 | -2 |
| 8 | [0,1,0,0,0]_{A5} | 2 | 0 | -2 |
| 8 | [1,1,0,0,1]_{A5} | 1 | 0 | -2 |
| 8 | [2,0,0,0,0]_{A5} | 1 | 0 | -2 |
| 8 | [0,0,0,0,0]_{A5} | 1 | 0 | 0 |
| 8 | [0,0,1,1,1]_{A5} | 2 | 0 | 0 |
| 8 | [0,0,2,0,0]_{A5} | 4 | 0 | 0 |
| 8 | [0,1,0,0,2]_{A5} | 2 | 0 | 0 |
| 8 | [0,1,0,1,0]_{A5} | 5 | 0 | 0 |
| 8 | [1,0,0,0,1]_{A5} | 5 | 0 | 0 |
| 8 | [1,1,1,0,0]_{A5} | 2 | 0 | 0 |
| 8 | [2,0,0,1,0]_{A5} | 2 | 0 | 0 |
| 8 | [0,0,0,0,2]_{A5} | 1 | 0 | 2 |
| 8 | [0,0,0,1,0]_{A5} | 2 | 0 | 2 |
| 8 | [1,0,0,1,1]_{A5} | 1 | 0 | 2 |
| 8 | [1,0,1,0,0]_{A5} | 2 | 0 | 2 |
| 9 | [1,0,0,0,1]_{A5} | 1 | 0 | -3 |
| 9 | [0,0,0,0,2]_{A5} | -2 | 0 | -1 |
| 9 | [0,0,0,1,0]_{A5} | -3 | 0 | -1 |
| 9 | [0,0,2,1,0]_{A5} | 1 | 0 | -1 |
| 9 | [0,1,1,0,1]_{A5} | 2 | 0 | -1 |
| 9 | [1,0,0,0,3]_{A5} | -1 | 0 | -1 |
| 9 | [1,0,0,1,1]_{A5} | -2 | 0 | -1 |
| 9 | [1,0,1,0,0]_{A5} | -1 | 0 | -1 |
| 9 | [2,0,1,0,1]_{A5} | -1 | 0 | -1 |
| 9 | [2,1,0,0,0]_{A5} | -2 | 0 | -1 |
| 9 | [4,0,0,0,0]_{A5} | -1 | 0 | -1 |
| 9 | [0,0,0,0,4]_{A5} | -1 | 0 | 1 |
| 9 | [0,0,0,1,2]_{A5} | -2 | 0 | 1 |
| 9 | [0,0,1,0,1]_{A5} | -1 | 0 | 1 |
| 9 | [0,1,0,0,0]_{A5} | -3 | 0 | 1 |
| 9 | [0,1,2,0,0]_{A5} | 1 | 0 | 1 |
| 9 | [1,0,1,0,2]_{A5} | -1 | 0 | 1 |
| 9 | [1,0,1,1,0]_{A5} | 2 | 0 | 1 |
| 9 | [1,1,0,0,1]_{A5} | -2 | 0 | 1 |
| 9 | [2,0,0,0,0]_{A5} | -2 | 0 | 1 |
| 9 | [3,0,0,0,1]_{A5} | -1 | 0 | 1 |
| 9 | [1,0,0,0,1]_{A5} | 1 | 0 | 3 |
| 10 | [0,0,0,1,2]_{A5} | -2 | 0 | -2 |
| 10 | [0,0,0,2,0]_{A5} | -1 | 0 | -2 |
| 10 | [0,0,1,0,1]_{A5} | -8 | 0 | -2 |
| 10 | [0,1,0,0,0]_{A5} | -6 | 0 | -2 |
| 10 | [1,0,1,0,2]_{A5} | -1 | 0 | -2 |
| 10 | [1,0,1,1,0]_{A5} | -2 | 0 | -2 |
| 10 | [1,1,0,0,1]_{A5} | -6 | 0 | -2 |
| 10 | [2,0,0,0,0]_{A5} | -5 | 0 | -2 |
| 10 | [2,1,0,1,0]_{A5} | -1 | 0 | -2 |
| 10 | [3,0,0,0,1]_{A5} | -1 | 0 | -2 |
| 10 | [0,0,0,0,0]_{A5} | -7 | 0 | 0 |
| 10 | [0,0,0,2,2]_{A5} | -1 | 0 | 0 |
| 10 | [0,0,0,3,0]_{A5} | -1 | 0 | 0 |
| 10 | [0,0,1,0,3]_{A5} | -3 | 0 | 0 |
| 10 | [0,0,1,1,1]_{A5} | -11 | 0 | 0 |
| 10 | [0,0,2,0,0]_{A5} | -10 | 0 | 0 |
| 10 | [0,1,0,0,2]_{A5} | -12 | 0 | 0 |
| 10 | [0,1,0,1,0]_{A5} | -21 | 0 | 0 |
| 10 | [0,3,0,0,0]_{A5} | -1 | 0 | 0 |
| 10 | [1,0,0,0,1]_{A5} | -21 | 0 | 0 |
| 10 | [1,0,2,0,1]_{A5} | -2 | 0 | 0 |
| 10 | [1,1,0,1,1]_{A5} | -4 | 0 | 0 |
| 10 | [1,1,1,0,0]_{A5} | -11 | 0 | 0 |
| 10 | [2,0,0,0,2]_{A5} | -4 | 0 | 0 |
| 10 | [2,0,0,1,0]_{A5} | -12 | 0 | 0 |
| 10 | [2,2,0,0,0]_{A5} | -1 | 0 | 0 |
| 10 | [3,0,1,0,0]_{A5} | -3 | 0 | 0 |
| 10 | [0,0,0,0,2]_{A5} | -5 | 0 | 2 |
| 10 | [0,0,0,1,0]_{A5} | -6 | 0 | 2 |
| 10 | [0,1,0,1,2]_{A5} | -1 | 0 | 2 |
| 10 | [0,1,1,0,1]_{A5} | -2 | 0 | 2 |
| 10 | [0,2,0,0,0]_{A5} | -1 | 0 | 2 |
| 10 | [1,0,0,0,3]_{A5} | -1 | 0 | 2 |
| 10 | [1,0,0,1,1]_{A5} | -6 | 0 | 2 |
| 10 | [1,0,1,0,0]_{A5} | -8 | 0 | 2 |
| 10 | [2,0,1,0,1]_{A5} | -1 | 0 | 2 |
| 10 | [2,1,0,0,0]_{A5} | -2 | 0 | 2 |

## Appendix F. Complete reconstruction checks

```json
{
  "maximum_t_degree": 10,
  "theory_id": "su3_nf5_k3o2_infinite",
  "validation_results": {
    "all_passed": true,
    "difference_is_empty": true,
    "independent_scalar_pe_equal": true,
    "multiplicities_integral": true,
    "multiplicities_nonnegative": true,
    "order_stability": true,
    "q_refined_dimensions_equal": true,
    "refined_character_series_equal": true,
    "unrefined_dimensions_equal": true
  }
}
```

## Appendix G. Complete operator-content tables

| degree | representation | mult. | dimension | x | q | source/classification |
|---|---|---|---|---|---|---|
| 2 | [0,0,0,0,0]_{A5} | 1 | 1 | 0 | 0 | low_degree_generator_candidate |
| 2 | [1,0,0,0,1]_{A5} | 1 | 35 | 0 | 0 | low_degree_generator_candidate |
| 3 | [0,0,0,1,0]_{A5} | 1 | 15 | 0 | -1 | low_degree_generator_candidate |
| 3 | [0,1,0,0,0]_{A5} | 1 | 15 | 0 | 1 | low_degree_generator_candidate |

| degree | representation | mult. | dimension | x | q | source/classification |
|---|---|---|---|---|---|---|
| 4 | [0,0,0,0,0]_{A5} | -1 | 1 | 0 | 0 | first_relation_candidate |
| 4 | [1,0,0,0,1]_{A5} | -1 | 35 | 0 | 0 | first_relation_candidate |

## Appendix H. Complete branched character series

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 0 | [0, 0, 0, 0] | 1 | 0 | 0 |
| 2 | [0, 0, 0, 0] | 2 | 0 | 0 |
| 2 | [0, 0, 0, 1] | 1 | -6 | 0 |
| 2 | [1, 0, 0, 0] | 1 | 6 | 0 |
| 2 | [1, 0, 0, 1] | 1 | 0 | 0 |
| 3 | [0, 0, 0, 1] | 1 | 4 | -1 |
| 3 | [0, 0, 1, 0] | 1 | -2 | -1 |
| 3 | [0, 1, 0, 0] | 1 | 2 | 1 |
| 3 | [1, 0, 0, 0] | 1 | -4 | 1 |
| 4 | [0, 0, 0, 0] | 3 | 0 | 0 |
| 4 | [0, 0, 0, 1] | 2 | -6 | 0 |
| 4 | [0, 0, 0, 2] | 1 | -12 | 0 |
| 4 | [0, 1, 0, 1] | 1 | 6 | 0 |
| 4 | [0, 1, 1, 0] | 1 | 0 | 0 |
| 4 | [1, 0, 0, 0] | 2 | 6 | 0 |
| 4 | [1, 0, 0, 1] | 3 | 0 | 0 |
| 4 | [1, 0, 0, 2] | 1 | -6 | 0 |
| 4 | [1, 0, 1, 0] | 1 | -6 | 0 |
| 4 | [2, 0, 0, 0] | 1 | 12 | 0 |
| 4 | [2, 0, 0, 1] | 1 | 6 | 0 |
| 4 | [2, 0, 0, 2] | 1 | 0 | 0 |
| 5 | [0, 0, 0, 1] | 2 | 4 | -1 |
| 5 | [0, 0, 0, 2] | 1 | -2 | -1 |
| 5 | [0, 0, 1, 0] | 2 | -2 | -1 |
| 5 | [0, 0, 1, 1] | 1 | -8 | -1 |
| 5 | [0, 1, 0, 0] | 2 | 2 | 1 |
| 5 | [0, 1, 0, 1] | 1 | -4 | 1 |
| 5 | [1, 0, 0, 0] | 2 | -4 | 1 |
| 5 | [1, 0, 0, 1] | 1 | -10 | 1 |
| 5 | [1, 0, 0, 1] | 1 | 10 | -1 |
| 5 | [1, 0, 0, 2] | 1 | 4 | -1 |
| 5 | [1, 0, 1, 0] | 1 | 4 | -1 |
| 5 | [1, 0, 1, 1] | 1 | -2 | -1 |
| 5 | [1, 1, 0, 0] | 1 | 8 | 1 |
| 5 | [1, 1, 0, 1] | 1 | 2 | 1 |
| 5 | [2, 0, 0, 0] | 1 | 2 | 1 |
| 5 | [2, 0, 0, 1] | 1 | -4 | 1 |
| 6 | [0, 0, 0, 0] | 4 | 0 | 0 |
| 6 | [0, 0, 0, 1] | 3 | -6 | 0 |
| 6 | [0, 0, 0, 2] | 2 | -12 | 0 |
| 6 | [0, 0, 0, 2] | 1 | 8 | -2 |
| 6 | [0, 0, 0, 3] | 1 | -18 | 0 |
| 6 | [0, 0, 1, 1] | 1 | 2 | -2 |
| 6 | [0, 0, 2, 0] | 1 | -4 | -2 |
| 6 | [0, 1, 0, 1] | 2 | 6 | 0 |
| 6 | [0, 1, 0, 2] | 1 | 0 | 0 |
| 6 | [0, 1, 1, 0] | 2 | 0 | 0 |
| 6 | [0, 1, 1, 1] | 1 | -6 | 0 |
| 6 | [0, 2, 0, 0] | 1 | 4 | 2 |
| 6 | [1, 0, 0, 0] | 3 | 6 | 0 |
| 6 | [1, 0, 0, 1] | 5 | 0 | 0 |
| 6 | [1, 0, 0, 2] | 3 | -6 | 0 |
| 6 | [1, 0, 0, 3] | 1 | -12 | 0 |
| 6 | [1, 0, 1, 0] | 2 | -6 | 0 |
| 6 | [1, 0, 1, 1] | 1 | -12 | 0 |
| 6 | [1, 1, 0, 0] | 1 | -2 | 2 |
| 6 | [1, 1, 0, 1] | 1 | 12 | 0 |
| 6 | [1, 1, 0, 2] | 1 | 6 | 0 |
| 6 | [1, 1, 1, 0] | 1 | 6 | 0 |
| 6 | [1, 1, 1, 1] | 1 | 0 | 0 |
| 6 | [2, 0, 0, 0] | 1 | -8 | 2 |
| 6 | [2, 0, 0, 0] | 2 | 12 | 0 |
| 6 | [2, 0, 0, 1] | 3 | 6 | 0 |
| 6 | [2, 0, 0, 2] | 3 | 0 | 0 |
| 6 | [2, 0, 0, 3] | 1 | -6 | 0 |
| 6 | [2, 0, 1, 0] | 1 | 0 | 0 |
| 6 | [2, 0, 1, 1] | 1 | -6 | 0 |
| 6 | [3, 0, 0, 0] | 1 | 18 | 0 |
| 6 | [3, 0, 0, 1] | 1 | 12 | 0 |
| 6 | [3, 0, 0, 2] | 1 | 6 | 0 |
| 6 | [3, 0, 0, 3] | 1 | 0 | 0 |
| 7 | [0, 0, 0, 1] | 3 | 4 | -1 |
| 7 | [0, 0, 0, 2] | 2 | -2 | -1 |
| 7 | [0, 0, 0, 3] | 1 | -8 | -1 |
| 7 | [0, 0, 1, 0] | 3 | -2 | -1 |
| 7 | [0, 0, 1, 1] | 2 | -8 | -1 |
| 7 | [0, 0, 1, 2] | 1 | -14 | -1 |
| 7 | [0, 1, 0, 0] | 3 | 2 | 1 |
| 7 | [0, 1, 0, 1] | 2 | -4 | 1 |
| 7 | [0, 1, 0, 2] | 1 | -10 | 1 |
| 7 | [0, 1, 0, 2] | 1 | 10 | -1 |
| 7 | [0, 1, 1, 1] | 1 | 4 | -1 |
| 7 | [0, 1, 2, 0] | 1 | -2 | -1 |
| 7 | [0, 2, 0, 1] | 1 | 8 | 1 |
| 7 | [0, 2, 1, 0] | 1 | 2 | 1 |
| 7 | [1, 0, 0, 0] | 3 | -4 | 1 |
| 7 | [1, 0, 0, 1] | 2 | -10 | 1 |
| 7 | [1, 0, 0, 1] | 2 | 10 | -1 |
| 7 | [1, 0, 0, 2] | 1 | -16 | 1 |
| 7 | [1, 0, 0, 2] | 3 | 4 | -1 |
| 7 | [1, 0, 0, 3] | 1 | -2 | -1 |
| 7 | [1, 0, 1, 0] | 2 | 4 | -1 |
| 7 | [1, 0, 1, 1] | 3 | -2 | -1 |
| 7 | [1, 0, 1, 2] | 1 | -8 | -1 |
| 7 | [1, 0, 2, 0] | 1 | -8 | -1 |
| 7 | [1, 1, 0, 0] | 2 | 8 | 1 |
| 7 | [1, 1, 0, 1] | 3 | 2 | 1 |
| 7 | [1, 1, 0, 2] | 1 | -4 | 1 |
| 7 | [1, 1, 1, 0] | 1 | -4 | 1 |
| 7 | [2, 0, 0, 0] | 2 | 2 | 1 |
| 7 | [2, 0, 0, 1] | 3 | -4 | 1 |
| 7 | [2, 0, 0, 1] | 1 | 16 | -1 |
| 7 | [2, 0, 0, 2] | 1 | -10 | 1 |
| 7 | [2, 0, 0, 2] | 1 | 10 | -1 |
| 7 | [2, 0, 0, 3] | 1 | 4 | -1 |
| 7 | [2, 0, 1, 0] | 1 | -10 | 1 |
| 7 | [2, 0, 1, 0] | 1 | 10 | -1 |
| 7 | [2, 0, 1, 1] | 1 | 4 | -1 |
| 7 | [2, 0, 1, 2] | 1 | -2 | -1 |
| 7 | [2, 1, 0, 0] | 1 | 14 | 1 |
| 7 | [2, 1, 0, 1] | 1 | 8 | 1 |
| 7 | [2, 1, 0, 2] | 1 | 2 | 1 |
| 7 | [3, 0, 0, 0] | 1 | 8 | 1 |
| 7 | [3, 0, 0, 1] | 1 | 2 | 1 |
| 7 | [3, 0, 0, 2] | 1 | -4 | 1 |
| 8 | [0, 0, 0, 0] | 5 | 0 | 0 |
| 8 | [0, 0, 0, 1] | 4 | -6 | 0 |
| 8 | [0, 0, 0, 2] | 3 | -12 | 0 |
| 8 | [0, 0, 0, 2] | 2 | 8 | -2 |
| 8 | [0, 0, 0, 3] | 2 | -18 | 0 |
| 8 | [0, 0, 0, 3] | 1 | 2 | -2 |
| 8 | [0, 0, 0, 4] | 1 | -24 | 0 |
| 8 | [0, 0, 1, 1] | 2 | 2 | -2 |
| 8 | [0, 0, 1, 2] | 1 | -4 | -2 |
| 8 | [0, 0, 2, 0] | 2 | -4 | -2 |
| 8 | [0, 0, 2, 1] | 1 | -10 | -2 |
| 8 | [0, 1, 0, 1] | 3 | 6 | 0 |
| 8 | [0, 1, 0, 2] | 2 | 0 | 0 |
| 8 | [0, 1, 0, 3] | 1 | -6 | 0 |
| 8 | [0, 1, 1, 0] | 3 | 0 | 0 |
| 8 | [0, 1, 1, 1] | 2 | -6 | 0 |
| 8 | [0, 1, 1, 2] | 1 | -12 | 0 |
| 8 | [0, 2, 0, 0] | 2 | 4 | 2 |
| 8 | [0, 2, 0, 1] | 1 | -2 | 2 |
| 8 | [0, 2, 0, 2] | 1 | 12 | 0 |
| 8 | [0, 2, 1, 1] | 1 | 6 | 0 |
| 8 | [0, 2, 2, 0] | 1 | 0 | 0 |
| 8 | [1, 0, 0, 0] | 4 | 6 | 0 |
| 8 | [1, 0, 0, 1] | 7 | 0 | 0 |
| 8 | [1, 0, 0, 2] | 5 | -6 | 0 |
| 8 | [1, 0, 0, 2] | 1 | 14 | -2 |
| 8 | [1, 0, 0, 3] | 3 | -12 | 0 |
| 8 | [1, 0, 0, 3] | 1 | 8 | -2 |
| 8 | [1, 0, 0, 4] | 1 | -18 | 0 |
| 8 | [1, 0, 1, 0] | 3 | -6 | 0 |
| 8 | [1, 0, 1, 1] | 2 | -12 | 0 |
| 8 | [1, 0, 1, 1] | 1 | 8 | -2 |
| 8 | [1, 0, 1, 2] | 1 | -18 | 0 |
| 8 | [1, 0, 1, 2] | 1 | 2 | -2 |
| 8 | [1, 0, 2, 0] | 1 | 2 | -2 |
| 8 | [1, 0, 2, 1] | 1 | -4 | -2 |
| 8 | [1, 1, 0, 0] | 2 | -2 | 2 |
| 8 | [1, 1, 0, 1] | 1 | -8 | 2 |
| 8 | [1, 1, 0, 1] | 2 | 12 | 0 |
| 8 | [1, 1, 0, 2] | 3 | 6 | 0 |
| 8 | [1, 1, 0, 3] | 1 | 0 | 0 |
| 8 | [1, 1, 1, 0] | 2 | 6 | 0 |
| 8 | [1, 1, 1, 1] | 3 | 0 | 0 |
| 8 | [1, 1, 1, 2] | 1 | -6 | 0 |
| 8 | [1, 1, 2, 0] | 1 | -6 | 0 |
| 8 | [1, 2, 0, 0] | 1 | 10 | 2 |
| 8 | [1, 2, 0, 1] | 1 | 4 | 2 |
| 8 | [2, 0, 0, 0] | 2 | -8 | 2 |
| 8 | [2, 0, 0, 0] | 3 | 12 | 0 |
| 8 | [2, 0, 0, 1] | 1 | -14 | 2 |
| 8 | [2, 0, 0, 1] | 5 | 6 | 0 |
| 8 | [2, 0, 0, 2] | 6 | 0 | 0 |
| 8 | [2, 0, 0, 3] | 3 | -6 | 0 |
| 8 | [2, 0, 0, 4] | 1 | -12 | 0 |
| 8 | [2, 0, 1, 0] | 2 | 0 | 0 |
| 8 | [2, 0, 1, 1] | 3 | -6 | 0 |
| 8 | [2, 0, 1, 2] | 1 | -12 | 0 |
| 8 | [2, 0, 2, 0] | 1 | -12 | 0 |
| 8 | [2, 1, 0, 0] | 1 | 4 | 2 |
| 8 | [2, 1, 0, 1] | 1 | -2 | 2 |
| 8 | [2, 1, 0, 1] | 1 | 18 | 0 |
| 8 | [2, 1, 0, 2] | 1 | 12 | 0 |
| 8 | [2, 1, 0, 3] | 1 | 6 | 0 |
| 8 | [2, 1, 1, 0] | 1 | 12 | 0 |
| 8 | [2, 1, 1, 1] | 1 | 6 | 0 |
| 8 | [2, 1, 1, 2] | 1 | 0 | 0 |
| 8 | [3, 0, 0, 0] | 1 | -2 | 2 |
| 8 | [3, 0, 0, 0] | 2 | 18 | 0 |
| 8 | [3, 0, 0, 1] | 1 | -8 | 2 |
| 8 | [3, 0, 0, 1] | 3 | 12 | 0 |
| 8 | [3, 0, 0, 2] | 3 | 6 | 0 |
| 8 | [3, 0, 0, 3] | 3 | 0 | 0 |
| 8 | [3, 0, 0, 4] | 1 | -6 | 0 |
| 8 | [3, 0, 1, 0] | 1 | 6 | 0 |
| 8 | [3, 0, 1, 1] | 1 | 0 | 0 |
| 8 | [3, 0, 1, 2] | 1 | -6 | 0 |
| 8 | [4, 0, 0, 0] | 1 | 24 | 0 |
| 8 | [4, 0, 0, 1] | 1 | 18 | 0 |
| 8 | [4, 0, 0, 2] | 1 | 12 | 0 |
| 8 | [4, 0, 0, 3] | 1 | 6 | 0 |
| 8 | [4, 0, 0, 4] | 1 | 0 | 0 |
| 9 | [0, 0, 0, 1] | 4 | 4 | -1 |
| 9 | [0, 0, 0, 2] | 3 | -2 | -1 |
| 9 | [0, 0, 0, 3] | 2 | -8 | -1 |
| 9 | [0, 0, 0, 3] | 1 | 12 | -3 |
| 9 | [0, 0, 0, 4] | 1 | -14 | -1 |
| 9 | [0, 0, 1, 0] | 4 | -2 | -1 |
| 9 | [0, 0, 1, 1] | 3 | -8 | -1 |
| 9 | [0, 0, 1, 2] | 2 | -14 | -1 |
| 9 | [0, 0, 1, 2] | 1 | 6 | -3 |
| 9 | [0, 0, 1, 3] | 1 | -20 | -1 |
| 9 | [0, 0, 2, 1] | 1 | 0 | -3 |
| 9 | [0, 0, 3, 0] | 1 | -6 | -3 |
| 9 | [0, 1, 0, 0] | 4 | 2 | 1 |
| 9 | [0, 1, 0, 1] | 3 | -4 | 1 |
| 9 | [0, 1, 0, 2] | 2 | -10 | 1 |
| 9 | [0, 1, 0, 2] | 2 | 10 | -1 |
| 9 | [0, 1, 0, 3] | 1 | -16 | 1 |
| 9 | [0, 1, 0, 3] | 1 | 4 | -1 |
| 9 | [0, 1, 1, 1] | 2 | 4 | -1 |
| 9 | [0, 1, 1, 2] | 1 | -2 | -1 |
| 9 | [0, 1, 2, 0] | 2 | -2 | -1 |
| 9 | [0, 1, 2, 1] | 1 | -8 | -1 |
| 9 | [0, 2, 0, 1] | 2 | 8 | 1 |
| 9 | [0, 2, 0, 2] | 1 | 2 | 1 |
| 9 | [0, 2, 1, 0] | 2 | 2 | 1 |
| 9 | [0, 2, 1, 1] | 1 | -4 | 1 |
| 9 | [0, 3, 0, 0] | 1 | 6 | 3 |
| 9 | [1, 0, 0, 0] | 4 | -4 | 1 |
| 9 | [1, 0, 0, 1] | 3 | -10 | 1 |
| 9 | [1, 0, 0, 1] | 3 | 10 | -1 |
| 9 | [1, 0, 0, 2] | 2 | -16 | 1 |
| 9 | [1, 0, 0, 2] | 5 | 4 | -1 |
| 9 | [1, 0, 0, 3] | 1 | -22 | 1 |
| 9 | [1, 0, 0, 3] | 3 | -2 | -1 |
| 9 | [1, 0, 0, 4] | 1 | -8 | -1 |
| 9 | [1, 0, 1, 0] | 3 | 4 | -1 |
| 9 | [1, 0, 1, 1] | 5 | -2 | -1 |
| 9 | [1, 0, 1, 2] | 3 | -8 | -1 |
| 9 | [1, 0, 1, 3] | 1 | -14 | -1 |
| 9 | [1, 0, 2, 0] | 2 | -8 | -1 |
| 9 | [1, 0, 2, 1] | 1 | -14 | -1 |
| 9 | [1, 1, 0, 0] | 3 | 8 | 1 |
| 9 | [1, 1, 0, 1] | 5 | 2 | 1 |
| 9 | [1, 1, 0, 2] | 3 | -4 | 1 |
| 9 | [1, 1, 0, 2] | 1 | 16 | -1 |
| 9 | [1, 1, 0, 3] | 1 | -10 | 1 |
| 9 | [1, 1, 0, 3] | 1 | 10 | -1 |
| 9 | [1, 1, 1, 0] | 2 | -4 | 1 |
| 9 | [1, 1, 1, 1] | 1 | -10 | 1 |
| 9 | [1, 1, 1, 1] | 1 | 10 | -1 |
| 9 | [1, 1, 1, 2] | 1 | 4 | -1 |
| 9 | [1, 1, 2, 0] | 1 | 4 | -1 |
| 9 | [1, 1, 2, 1] | 1 | -2 | -1 |
| 9 | [1, 2, 0, 0] | 1 | 0 | 3 |
| 9 | [1, 2, 0, 1] | 1 | 14 | 1 |
| 9 | [1, 2, 0, 2] | 1 | 8 | 1 |
| 9 | [1, 2, 1, 0] | 1 | 8 | 1 |
| 9 | [1, 2, 1, 1] | 1 | 2 | 1 |
| 9 | [2, 0, 0, 0] | 3 | 2 | 1 |
| 9 | [2, 0, 0, 1] | 5 | -4 | 1 |
| 9 | [2, 0, 0, 1] | 2 | 16 | -1 |
| 9 | [2, 0, 0, 2] | 3 | -10 | 1 |
| 9 | [2, 0, 0, 2] | 3 | 10 | -1 |
| 9 | [2, 0, 0, 3] | 1 | -16 | 1 |
| 9 | [2, 0, 0, 3] | 3 | 4 | -1 |
| 9 | [2, 0, 0, 4] | 1 | -2 | -1 |
| 9 | [2, 0, 1, 0] | 2 | -10 | 1 |
| 9 | [2, 0, 1, 0] | 2 | 10 | -1 |
| 9 | [2, 0, 1, 1] | 1 | -16 | 1 |
| 9 | [2, 0, 1, 1] | 3 | 4 | -1 |
| 9 | [2, 0, 1, 2] | 3 | -2 | -1 |
| 9 | [2, 0, 1, 3] | 1 | -8 | -1 |
| 9 | [2, 0, 2, 0] | 1 | -2 | -1 |
| 9 | [2, 0, 2, 1] | 1 | -8 | -1 |
| 9 | [2, 1, 0, 0] | 1 | -6 | 3 |
| 9 | [2, 1, 0, 0] | 2 | 14 | 1 |
| 9 | [2, 1, 0, 1] | 3 | 8 | 1 |
| 9 | [2, 1, 0, 2] | 3 | 2 | 1 |
| 9 | [2, 1, 0, 3] | 1 | -4 | 1 |
| 9 | [2, 1, 1, 0] | 1 | 2 | 1 |
| 9 | [2, 1, 1, 1] | 1 | -4 | 1 |
| 9 | [3, 0, 0, 0] | 1 | -12 | 3 |
| 9 | [3, 0, 0, 0] | 2 | 8 | 1 |
| 9 | [3, 0, 0, 1] | 3 | 2 | 1 |
| 9 | [3, 0, 0, 1] | 1 | 22 | -1 |
| 9 | [3, 0, 0, 2] | 3 | -4 | 1 |
| 9 | [3, 0, 0, 2] | 1 | 16 | -1 |
| 9 | [3, 0, 0, 3] | 1 | -10 | 1 |
| 9 | [3, 0, 0, 3] | 1 | 10 | -1 |
| 9 | [3, 0, 0, 4] | 1 | 4 | -1 |
| 9 | [3, 0, 1, 0] | 1 | -4 | 1 |
| 9 | [3, 0, 1, 0] | 1 | 16 | -1 |
| 9 | [3, 0, 1, 1] | 1 | -10 | 1 |
| 9 | [3, 0, 1, 1] | 1 | 10 | -1 |
| 9 | [3, 0, 1, 2] | 1 | 4 | -1 |
| 9 | [3, 0, 1, 3] | 1 | -2 | -1 |
| 9 | [3, 1, 0, 0] | 1 | 20 | 1 |
| 9 | [3, 1, 0, 1] | 1 | 14 | 1 |
| 9 | [3, 1, 0, 2] | 1 | 8 | 1 |
| 9 | [3, 1, 0, 3] | 1 | 2 | 1 |
| 9 | [4, 0, 0, 0] | 1 | 14 | 1 |
| 9 | [4, 0, 0, 1] | 1 | 8 | 1 |
| 9 | [4, 0, 0, 2] | 1 | 2 | 1 |
| 9 | [4, 0, 0, 3] | 1 | -4 | 1 |
| 10 | [0, 0, 0, 0] | 6 | 0 | 0 |
| 10 | [0, 0, 0, 1] | 5 | -6 | 0 |
| 10 | [0, 0, 0, 2] | 4 | -12 | 0 |
| 10 | [0, 0, 0, 2] | 3 | 8 | -2 |
| 10 | [0, 0, 0, 3] | 3 | -18 | 0 |
| 10 | [0, 0, 0, 3] | 2 | 2 | -2 |
| 10 | [0, 0, 0, 4] | 2 | -24 | 0 |
| 10 | [0, 0, 0, 4] | 1 | -4 | -2 |
| 10 | [0, 0, 0, 5] | 1 | -30 | 0 |
| 10 | [0, 0, 1, 1] | 3 | 2 | -2 |
| 10 | [0, 0, 1, 2] | 2 | -4 | -2 |
| 10 | [0, 0, 1, 3] | 1 | -10 | -2 |
| 10 | [0, 0, 2, 0] | 3 | -4 | -2 |
| 10 | [0, 0, 2, 1] | 2 | -10 | -2 |
| 10 | [0, 0, 2, 2] | 1 | -16 | -2 |
| 10 | [0, 1, 0, 1] | 4 | 6 | 0 |
| 10 | [0, 1, 0, 2] | 3 | 0 | 0 |
| 10 | [0, 1, 0, 3] | 2 | -6 | 0 |
| 10 | [0, 1, 0, 3] | 1 | 14 | -2 |
| 10 | [0, 1, 0, 4] | 1 | -12 | 0 |
| 10 | [0, 1, 1, 0] | 4 | 0 | 0 |
| 10 | [0, 1, 1, 1] | 3 | -6 | 0 |
| 10 | [0, 1, 1, 2] | 2 | -12 | 0 |
| 10 | [0, 1, 1, 2] | 1 | 8 | -2 |
| 10 | [0, 1, 1, 3] | 1 | -18 | 0 |
| 10 | [0, 1, 2, 1] | 1 | 2 | -2 |
| 10 | [0, 1, 3, 0] | 1 | -4 | -2 |
| 10 | [0, 2, 0, 0] | 3 | 4 | 2 |
| 10 | [0, 2, 0, 1] | 2 | -2 | 2 |
| 10 | [0, 2, 0, 2] | 1 | -8 | 2 |
| 10 | [0, 2, 0, 2] | 2 | 12 | 0 |
| 10 | [0, 2, 0, 3] | 1 | 6 | 0 |
| 10 | [0, 2, 1, 1] | 2 | 6 | 0 |
| 10 | [0, 2, 1, 2] | 1 | 0 | 0 |
| 10 | [0, 2, 2, 0] | 2 | 0 | 0 |
| 10 | [0, 2, 2, 1] | 1 | -6 | 0 |
| 10 | [0, 3, 0, 1] | 1 | 10 | 2 |
| 10 | [0, 3, 1, 0] | 1 | 4 | 2 |
| 10 | [1, 0, 0, 0] | 5 | 6 | 0 |
| 10 | [1, 0, 0, 1] | 9 | 0 | 0 |
| 10 | [1, 0, 0, 2] | 7 | -6 | 0 |
| 10 | [1, 0, 0, 2] | 2 | 14 | -2 |
| 10 | [1, 0, 0, 3] | 5 | -12 | 0 |
| 10 | [1, 0, 0, 3] | 3 | 8 | -2 |
| 10 | [1, 0, 0, 4] | 3 | -18 | 0 |
| 10 | [1, 0, 0, 4] | 1 | 2 | -2 |
| 10 | [1, 0, 0, 5] | 1 | -24 | 0 |
| 10 | [1, 0, 1, 0] | 4 | -6 | 0 |
| 10 | [1, 0, 1, 1] | 3 | -12 | 0 |
| 10 | [1, 0, 1, 1] | 2 | 8 | -2 |
| 10 | [1, 0, 1, 2] | 2 | -18 | 0 |
| 10 | [1, 0, 1, 2] | 3 | 2 | -2 |
| 10 | [1, 0, 1, 3] | 1 | -24 | 0 |
| 10 | [1, 0, 1, 3] | 1 | -4 | -2 |
| 10 | [1, 0, 2, 0] | 2 | 2 | -2 |
| 10 | [1, 0, 2, 1] | 3 | -4 | -2 |
| 10 | [1, 0, 2, 2] | 1 | -10 | -2 |
| 10 | [1, 0, 3, 0] | 1 | -10 | -2 |
| 10 | [1, 1, 0, 0] | 3 | -2 | 2 |
| 10 | [1, 1, 0, 1] | 2 | -8 | 2 |
| 10 | [1, 1, 0, 1] | 3 | 12 | 0 |
| 10 | [1, 1, 0, 2] | 1 | -14 | 2 |
| 10 | [1, 1, 0, 2] | 5 | 6 | 0 |
| 10 | [1, 1, 0, 3] | 3 | 0 | 0 |
| 10 | [1, 1, 0, 4] | 1 | -6 | 0 |
| 10 | [1, 1, 1, 0] | 3 | 6 | 0 |
| 10 | [1, 1, 1, 1] | 5 | 0 | 0 |
| 10 | [1, 1, 1, 2] | 3 | -6 | 0 |
| 10 | [1, 1, 1, 3] | 1 | -12 | 0 |
| 10 | [1, 1, 2, 0] | 2 | -6 | 0 |
| 10 | [1, 1, 2, 1] | 1 | -12 | 0 |
| 10 | [1, 2, 0, 0] | 2 | 10 | 2 |
| 10 | [1, 2, 0, 1] | 3 | 4 | 2 |
| 10 | [1, 2, 0, 2] | 1 | -2 | 2 |
| 10 | [1, 2, 0, 2] | 1 | 18 | 0 |
| 10 | [1, 2, 0, 3] | 1 | 12 | 0 |
| 10 | [1, 2, 1, 0] | 1 | -2 | 2 |
| 10 | [1, 2, 1, 1] | 1 | 12 | 0 |
| 10 | [1, 2, 1, 2] | 1 | 6 | 0 |
| 10 | [1, 2, 2, 0] | 1 | 6 | 0 |
| 10 | [1, 2, 2, 1] | 1 | 0 | 0 |
| 10 | [2, 0, 0, 0] | 3 | -8 | 2 |
| 10 | [2, 0, 0, 0] | 4 | 12 | 0 |
| 10 | [2, 0, 0, 1] | 2 | -14 | 2 |
| 10 | [2, 0, 0, 1] | 7 | 6 | 0 |
| 10 | [2, 0, 0, 2] | 1 | -20 | 2 |
| 10 | [2, 0, 0, 2] | 9 | 0 | 0 |
| 10 | [2, 0, 0, 2] | 1 | 20 | -2 |
| 10 | [2, 0, 0, 3] | 6 | -6 | 0 |
| 10 | [2, 0, 0, 3] | 1 | 14 | -2 |
| 10 | [2, 0, 0, 4] | 3 | -12 | 0 |
| 10 | [2, 0, 0, 4] | 1 | 8 | -2 |
| 10 | [2, 0, 0, 5] | 1 | -18 | 0 |
| 10 | [2, 0, 1, 0] | 3 | 0 | 0 |
| 10 | [2, 0, 1, 1] | 5 | -6 | 0 |
| 10 | [2, 0, 1, 1] | 1 | 14 | -2 |
| 10 | [2, 0, 1, 2] | 3 | -12 | 0 |
| 10 | [2, 0, 1, 2] | 1 | 8 | -2 |
| 10 | [2, 0, 1, 3] | 1 | -18 | 0 |
| 10 | [2, 0, 1, 3] | 1 | 2 | -2 |
| 10 | [2, 0, 2, 0] | 2 | -12 | 0 |
| 10 | [2, 0, 2, 0] | 1 | 8 | -2 |
| 10 | [2, 0, 2, 1] | 1 | -18 | 0 |
| 10 | [2, 0, 2, 1] | 1 | 2 | -2 |
| 10 | [2, 0, 2, 2] | 1 | -4 | -2 |
| 10 | [2, 1, 0, 0] | 2 | 4 | 2 |
| 10 | [2, 1, 0, 1] | 3 | -2 | 2 |
| 10 | [2, 1, 0, 1] | 2 | 18 | 0 |
| 10 | [2, 1, 0, 2] | 1 | -8 | 2 |
| 10 | [2, 1, 0, 2] | 3 | 12 | 0 |
| 10 | [2, 1, 0, 3] | 3 | 6 | 0 |
| 10 | [2, 1, 0, 4] | 1 | 0 | 0 |
| 10 | [2, 1, 1, 0] | 1 | -8 | 2 |
| 10 | [2, 1, 1, 0] | 2 | 12 | 0 |
| 10 | [2, 1, 1, 1] | 3 | 6 | 0 |
| 10 | [2, 1, 1, 2] | 3 | 0 | 0 |
| 10 | [2, 1, 1, 3] | 1 | -6 | 0 |
| 10 | [2, 1, 2, 0] | 1 | 0 | 0 |
| 10 | [2, 1, 2, 1] | 1 | -6 | 0 |
| 10 | [2, 2, 0, 0] | 1 | 16 | 2 |
| 10 | [2, 2, 0, 1] | 1 | 10 | 2 |
| 10 | [2, 2, 0, 2] | 1 | 4 | 2 |
| 10 | [3, 0, 0, 0] | 2 | -2 | 2 |
| 10 | [3, 0, 0, 0] | 3 | 18 | 0 |
| 10 | [3, 0, 0, 1] | 3 | -8 | 2 |
| 10 | [3, 0, 0, 1] | 5 | 12 | 0 |
| 10 | [3, 0, 0, 2] | 1 | -14 | 2 |
| 10 | [3, 0, 0, 2] | 6 | 6 | 0 |
| 10 | [3, 0, 0, 3] | 6 | 0 | 0 |
| 10 | [3, 0, 0, 4] | 3 | -6 | 0 |
| 10 | [3, 0, 0, 5] | 1 | -12 | 0 |
| 10 | [3, 0, 1, 0] | 1 | -14 | 2 |
| 10 | [3, 0, 1, 0] | 2 | 6 | 0 |
| 10 | [3, 0, 1, 1] | 3 | 0 | 0 |
| 10 | [3, 0, 1, 2] | 3 | -6 | 0 |
| 10 | [3, 0, 1, 3] | 1 | -12 | 0 |
| 10 | [3, 0, 2, 0] | 1 | -6 | 0 |
| 10 | [3, 0, 2, 1] | 1 | -12 | 0 |
| 10 | [3, 1, 0, 0] | 1 | 10 | 2 |
| 10 | [3, 1, 0, 1] | 1 | 4 | 2 |
| 10 | [3, 1, 0, 1] | 1 | 24 | 0 |
| 10 | [3, 1, 0, 2] | 1 | -2 | 2 |
| 10 | [3, 1, 0, 2] | 1 | 18 | 0 |
| 10 | [3, 1, 0, 3] | 1 | 12 | 0 |
| 10 | [3, 1, 0, 4] | 1 | 6 | 0 |
| 10 | [3, 1, 1, 0] | 1 | 18 | 0 |
| 10 | [3, 1, 1, 1] | 1 | 12 | 0 |
| 10 | [3, 1, 1, 2] | 1 | 6 | 0 |
| 10 | [3, 1, 1, 3] | 1 | 0 | 0 |
| 10 | [4, 0, 0, 0] | 1 | 4 | 2 |
| 10 | [4, 0, 0, 0] | 2 | 24 | 0 |
| 10 | [4, 0, 0, 1] | 1 | -2 | 2 |
| 10 | [4, 0, 0, 1] | 3 | 18 | 0 |
| 10 | [4, 0, 0, 2] | 1 | -8 | 2 |
| 10 | [4, 0, 0, 2] | 3 | 12 | 0 |
| 10 | [4, 0, 0, 3] | 3 | 6 | 0 |
| 10 | [4, 0, 0, 4] | 3 | 0 | 0 |
| 10 | [4, 0, 0, 5] | 1 | -6 | 0 |
| 10 | [4, 0, 1, 0] | 1 | 12 | 0 |
| 10 | [4, 0, 1, 1] | 1 | 6 | 0 |
| 10 | [4, 0, 1, 2] | 1 | 0 | 0 |
| 10 | [4, 0, 1, 3] | 1 | -6 | 0 |
| 10 | [5, 0, 0, 0] | 1 | 30 | 0 |
| 10 | [5, 0, 0, 1] | 1 | 24 | 0 |
| 10 | [5, 0, 0, 2] | 1 | 18 | 0 |
| 10 | [5, 0, 0, 3] | 1 | 12 | 0 |
| 10 | [5, 0, 0, 4] | 1 | 6 | 0 |
| 10 | [5, 0, 0, 5] | 1 | 0 | 0 |

## Appendix I. Complete branched refined PL

| degree | representation | multiplicity | x | q |
|---|---|---|---|---|
| 2 | [0, 0, 0, 0] | 2 | 0 | 0 |
| 2 | [0, 0, 0, 1] | 1 | -6 | 0 |
| 2 | [1, 0, 0, 0] | 1 | 6 | 0 |
| 2 | [1, 0, 0, 1] | 1 | 0 | 0 |
| 3 | [0, 0, 0, 1] | 1 | 4 | -1 |
| 3 | [0, 0, 1, 0] | 1 | -2 | -1 |
| 3 | [0, 1, 0, 0] | 1 | 2 | 1 |
| 3 | [1, 0, 0, 0] | 1 | -4 | 1 |
| 4 | [0, 0, 0, 0] | -2 | 0 | 0 |
| 4 | [0, 0, 0, 1] | -1 | -6 | 0 |
| 4 | [1, 0, 0, 0] | -1 | 6 | 0 |
| 4 | [1, 0, 0, 1] | -1 | 0 | 0 |
| 5 | [0, 0, 0, 0] | -1 | -10 | 1 |
| 5 | [0, 0, 0, 0] | -1 | 10 | -1 |
| 5 | [0, 0, 0, 1] | -2 | 4 | -1 |
| 5 | [0, 0, 0, 2] | -1 | -2 | -1 |
| 5 | [0, 0, 1, 0] | -2 | -2 | -1 |
| 5 | [0, 0, 1, 0] | -1 | 8 | 1 |
| 5 | [0, 0, 1, 1] | -1 | 2 | 1 |
| 5 | [0, 1, 0, 0] | -1 | -8 | -1 |
| 5 | [0, 1, 0, 0] | -2 | 2 | 1 |
| 5 | [0, 1, 0, 1] | -1 | -4 | 1 |
| 5 | [1, 0, 0, 0] | -2 | -4 | 1 |
| 5 | [1, 0, 1, 0] | -1 | 4 | -1 |
| 5 | [1, 1, 0, 0] | -1 | -2 | -1 |
| 5 | [2, 0, 0, 0] | -1 | 2 | 1 |
| 6 | [0, 0, 0, 0] | -1 | 0 | 0 |
| 6 | [0, 0, 0, 1] | -1 | 4 | 2 |
| 6 | [0, 0, 1, 0] | -1 | -2 | 2 |
| 6 | [0, 0, 2, 0] | -1 | 6 | 0 |
| 6 | [0, 1, 0, 0] | -1 | 2 | -2 |
| 6 | [0, 1, 0, 1] | -1 | 6 | 0 |
| 6 | [0, 1, 1, 0] | -2 | 0 | 0 |
| 6 | [0, 2, 0, 0] | -1 | -6 | 0 |
| 6 | [1, 0, 0, 0] | -1 | -4 | -2 |
| 6 | [1, 0, 0, 1] | -1 | 0 | 0 |
| 6 | [1, 0, 1, 0] | -1 | -6 | 0 |
| 7 | [0, 0, 0, 0] | 2 | -10 | 1 |
| 7 | [0, 0, 0, 0] | 2 | 10 | -1 |
| 7 | [0, 0, 0, 1] | 5 | 4 | -1 |
| 7 | [0, 0, 0, 1] | 1 | 14 | 1 |
| 7 | [0, 0, 0, 2] | 3 | -2 | -1 |
| 7 | [0, 0, 0, 2] | 1 | 8 | 1 |
| 7 | [0, 0, 0, 3] | 1 | 2 | 1 |
| 7 | [0, 0, 1, 0] | 5 | -2 | -1 |
| 7 | [0, 0, 1, 0] | 3 | 8 | 1 |
| 7 | [0, 0, 1, 1] | 1 | -8 | -1 |
| 7 | [0, 0, 1, 1] | 3 | 2 | 1 |
| 7 | [0, 0, 1, 2] | 1 | -4 | 1 |
| 7 | [0, 1, 0, 0] | 3 | -8 | -1 |
| 7 | [0, 1, 0, 0] | 5 | 2 | 1 |
| 7 | [0, 1, 0, 1] | 3 | -4 | 1 |
| 7 | [1, 0, 0, 0] | 1 | -14 | -1 |
| 7 | [1, 0, 0, 0] | 5 | -4 | 1 |
| 7 | [1, 0, 0, 1] | 1 | -10 | 1 |
| 7 | [1, 0, 0, 1] | 1 | 10 | -1 |
| 7 | [1, 0, 0, 2] | 1 | 4 | -1 |
| 7 | [1, 0, 1, 0] | 3 | 4 | -1 |
| 7 | [1, 0, 1, 1] | 1 | -2 | -1 |
| 7 | [1, 1, 0, 0] | 3 | -2 | -1 |
| 7 | [1, 1, 0, 0] | 1 | 8 | 1 |
| 7 | [1, 1, 0, 1] | 1 | 2 | 1 |
| 7 | [2, 0, 0, 0] | 1 | -8 | -1 |
| 7 | [2, 0, 0, 0] | 3 | 2 | 1 |
| 7 | [2, 0, 0, 1] | 1 | -4 | 1 |
| 7 | [2, 1, 0, 0] | 1 | 4 | -1 |
| 7 | [3, 0, 0, 0] | 1 | -2 | -1 |
| 8 | [0, 0, 0, 0] | 1 | -10 | -2 |
| 8 | [0, 0, 0, 0] | 6 | 0 | 0 |
| 8 | [0, 0, 0, 0] | 1 | 10 | 2 |
| 8 | [0, 0, 0, 1] | 7 | -6 | 0 |
| 8 | [0, 0, 0, 1] | 4 | 4 | 2 |
| 8 | [0, 0, 0, 2] | 2 | -2 | 2 |
| 8 | [0, 0, 1, 0] | 2 | -12 | 0 |
| 8 | [0, 0, 1, 0] | 5 | -2 | 2 |
| 8 | [0, 0, 1, 0] | 2 | 8 | -2 |
| 8 | [0, 0, 1, 1] | 1 | -8 | 2 |
| 8 | [0, 0, 1, 1] | 2 | 2 | -2 |
| 8 | [0, 0, 1, 1] | 2 | 12 | 0 |
| 8 | [0, 0, 1, 2] | 2 | 6 | 0 |
| 8 | [0, 0, 2, 0] | 6 | 6 | 0 |
| 8 | [0, 0, 2, 1] | 2 | 0 | 0 |
| 8 | [0, 1, 0, 0] | 2 | -8 | 2 |
| 8 | [0, 1, 0, 0] | 5 | 2 | -2 |
| 8 | [0, 1, 0, 0] | 2 | 12 | 0 |
| 8 | [0, 1, 0, 1] | 3 | -4 | -2 |
| 8 | [0, 1, 0, 1] | 9 | 6 | 0 |
| 8 | [0, 1, 0, 2] | 4 | 0 | 0 |
| 8 | [0, 1, 1, 0] | 13 | 0 | 0 |
| 8 | [0, 1, 1, 1] | 2 | -6 | 0 |
| 8 | [0, 2, 0, 0] | 6 | -6 | 0 |
| 8 | [1, 0, 0, 0] | 4 | -4 | -2 |
| 8 | [1, 0, 0, 0] | 7 | 6 | 0 |
| 8 | [1, 0, 0, 1] | 1 | -10 | -2 |
| 8 | [1, 0, 0, 1] | 14 | 0 | 0 |
| 8 | [1, 0, 0, 1] | 1 | 10 | 2 |
| 8 | [1, 0, 0, 2] | 2 | -6 | 0 |
| 8 | [1, 0, 0, 2] | 1 | 4 | 2 |
| 8 | [1, 0, 1, 0] | 9 | -6 | 0 |
| 8 | [1, 0, 1, 0] | 3 | 4 | 2 |
| 8 | [1, 0, 1, 1] | 1 | -2 | 2 |
| 8 | [1, 1, 0, 0] | 2 | -12 | 0 |
| 8 | [1, 1, 0, 0] | 2 | -2 | 2 |
| 8 | [1, 1, 0, 0] | 1 | 8 | -2 |
| 8 | [1, 1, 0, 1] | 1 | 2 | -2 |
| 8 | [1, 1, 1, 0] | 2 | 6 | 0 |
| 8 | [1, 2, 0, 0] | 2 | 0 | 0 |
| 8 | [2, 0, 0, 0] | 2 | 2 | -2 |
| 8 | [2, 0, 0, 1] | 1 | -4 | -2 |
| 8 | [2, 0, 0, 1] | 2 | 6 | 0 |
| 8 | [2, 0, 1, 0] | 4 | 0 | 0 |
| 8 | [2, 1, 0, 0] | 2 | -6 | 0 |
| 9 | [0, 0, 0, 0] | -1 | -20 | -1 |
| 9 | [0, 0, 0, 0] | -3 | -10 | 1 |
| 9 | [0, 0, 0, 0] | 1 | 0 | -3 |
| 9 | [0, 0, 0, 0] | 1 | 0 | 3 |
| 9 | [0, 0, 0, 0] | -3 | 10 | -1 |
| 9 | [0, 0, 0, 0] | -1 | 20 | 1 |
| 9 | [0, 0, 0, 1] | -1 | -16 | 1 |
| 9 | [0, 0, 0, 1] | 1 | -6 | -3 |
| 9 | [0, 0, 0, 1] | 1 | -6 | 3 |
| 9 | [0, 0, 0, 1] | -8 | 4 | -1 |
| 9 | [0, 0, 0, 1] | -3 | 14 | 1 |
| 9 | [0, 0, 0, 2] | -5 | -2 | -1 |
| 9 | [0, 0, 0, 2] | -3 | 8 | 1 |
| 9 | [0, 0, 0, 3] | -1 | -8 | -1 |
| 9 | [0, 0, 0, 3] | -3 | 2 | 1 |
| 9 | [0, 0, 0, 4] | -1 | -4 | 1 |
| 9 | [0, 0, 1, 0] | -7 | -2 | -1 |
| 9 | [0, 0, 1, 0] | -4 | 8 | 1 |
| 9 | [0, 0, 1, 1] | -3 | -8 | -1 |
| 9 | [0, 0, 1, 1] | -2 | 2 | 1 |
| 9 | [0, 0, 1, 2] | -3 | -4 | 1 |
| 9 | [0, 0, 2, 0] | 2 | -4 | 1 |
| 9 | [0, 0, 2, 1] | 1 | 10 | -1 |
| 9 | [0, 0, 3, 0] | 1 | 4 | -1 |
| 9 | [0, 1, 0, 0] | -4 | -8 | -1 |
| 9 | [0, 1, 0, 0] | -7 | 2 | 1 |
| 9 | [0, 1, 0, 1] | -1 | -14 | -1 |
| 9 | [0, 1, 0, 1] | -2 | -4 | 1 |
| 9 | [0, 1, 0, 2] | -1 | -10 | 1 |
| 9 | [0, 1, 1, 0] | 2 | -10 | 1 |
| 9 | [0, 1, 1, 0] | 2 | 10 | -1 |
| 9 | [0, 1, 1, 1] | 3 | 4 | -1 |
| 9 | [0, 1, 2, 0] | 1 | -2 | -1 |
| 9 | [0, 1, 2, 0] | 1 | 8 | 1 |
| 9 | [0, 2, 0, 0] | 2 | 4 | -1 |
| 9 | [0, 2, 0, 1] | 3 | -2 | -1 |
| 9 | [0, 2, 1, 0] | 1 | -8 | -1 |
| 9 | [0, 2, 1, 0] | 1 | 2 | 1 |
| 9 | [0, 3, 0, 0] | 1 | -4 | 1 |
| 9 | [1, 0, 0, 0] | -3 | -14 | -1 |
| 9 | [1, 0, 0, 0] | -8 | -4 | 1 |
| 9 | [1, 0, 0, 0] | 1 | 6 | -3 |
| 9 | [1, 0, 0, 0] | 1 | 6 | 3 |
| 9 | [1, 0, 0, 0] | -1 | 16 | -1 |
| 9 | [1, 0, 0, 1] | -3 | -10 | 1 |
| 9 | [1, 0, 0, 1] | 1 | 0 | -3 |
| 9 | [1, 0, 0, 1] | 1 | 0 | 3 |
| 9 | [1, 0, 0, 1] | -3 | 10 | -1 |
| 9 | [1, 0, 0, 2] | -3 | 4 | -1 |
| 9 | [1, 0, 0, 3] | -1 | -2 | -1 |
| 9 | [1, 0, 1, 0] | -2 | 4 | -1 |
| 9 | [1, 0, 1, 0] | -1 | 14 | 1 |
| 9 | [1, 0, 1, 1] | -1 | -2 | -1 |
| 9 | [1, 0, 1, 1] | 1 | 8 | 1 |
| 9 | [1, 0, 1, 2] | -1 | 2 | 1 |
| 9 | [1, 0, 2, 0] | 3 | 2 | 1 |
| 9 | [1, 1, 0, 0] | -2 | -2 | -1 |
| 9 | [1, 1, 0, 0] | -3 | 8 | 1 |
| 9 | [1, 1, 0, 1] | 1 | -8 | -1 |
| 9 | [1, 1, 0, 1] | -1 | 2 | 1 |
| 9 | [1, 1, 0, 2] | -1 | -4 | 1 |
| 9 | [1, 1, 1, 0] | 3 | -4 | 1 |
| 9 | [1, 2, 0, 0] | 1 | -10 | 1 |
| 9 | [2, 0, 0, 0] | -3 | -8 | -1 |
| 9 | [2, 0, 0, 0] | -5 | 2 | 1 |
| 9 | [2, 0, 0, 1] | -3 | -4 | 1 |
| 9 | [2, 0, 1, 0] | -1 | 10 | -1 |
| 9 | [2, 0, 1, 1] | -1 | 4 | -1 |
| 9 | [2, 1, 0, 0] | -3 | 4 | -1 |
| 9 | [2, 1, 0, 1] | -1 | -2 | -1 |
| 9 | [3, 0, 0, 0] | -3 | -2 | -1 |
| 9 | [3, 0, 0, 0] | -1 | 8 | 1 |
| 9 | [3, 0, 0, 1] | -1 | 2 | 1 |
| 9 | [4, 0, 0, 0] | -1 | 4 | -1 |
| 10 | [0, 0, 0, 0] | -6 | -10 | -2 |
| 10 | [0, 0, 0, 0] | -32 | 0 | 0 |
| 10 | [0, 0, 0, 0] | -6 | 10 | 2 |
| 10 | [0, 0, 0, 1] | -1 | -16 | -2 |
| 10 | [0, 0, 0, 1] | -37 | -6 | 0 |
| 10 | [0, 0, 0, 1] | -18 | 4 | 2 |
| 10 | [0, 0, 0, 1] | -2 | 14 | -2 |
| 10 | [0, 0, 0, 2] | -4 | -12 | 0 |
| 10 | [0, 0, 0, 2] | -12 | -2 | 2 |
| 10 | [0, 0, 0, 2] | -3 | 8 | -2 |
| 10 | [0, 0, 0, 2] | -1 | 18 | 0 |
| 10 | [0, 0, 0, 3] | -1 | -8 | 2 |
| 10 | [0, 0, 0, 3] | -2 | 2 | -2 |
| 10 | [0, 0, 0, 3] | -2 | 12 | 0 |
| 10 | [0, 0, 0, 4] | -1 | 6 | 0 |
| 10 | [0, 0, 1, 0] | -15 | -12 | 0 |
| 10 | [0, 0, 1, 0] | -21 | -2 | 2 |
| 10 | [0, 0, 1, 0] | -11 | 8 | -2 |
| 10 | [0, 0, 1, 0] | -3 | 18 | 0 |
| 10 | [0, 0, 1, 1] | -7 | -8 | 2 |
| 10 | [0, 0, 1, 1] | -14 | 2 | -2 |
| 10 | [0, 0, 1, 1] | -15 | 12 | 0 |
| 10 | [0, 0, 1, 2] | -3 | -4 | -2 |
| 10 | [0, 0, 1, 2] | -16 | 6 | 0 |
| 10 | [0, 0, 1, 3] | -4 | 0 | 0 |
| 10 | [0, 0, 2, 0] | -3 | -4 | -2 |
| 10 | [0, 0, 2, 0] | -24 | 6 | 0 |
| 10 | [0, 0, 2, 1] | -15 | 0 | 0 |
| 10 | [0, 0, 2, 2] | -1 | -6 | 0 |
| 10 | [0, 0, 3, 0] | -1 | -6 | 0 |
| 10 | [0, 1, 0, 0] | -3 | -18 | 0 |
| 10 | [0, 1, 0, 0] | -11 | -8 | 2 |
| 10 | [0, 1, 0, 0] | -21 | 2 | -2 |
| 10 | [0, 1, 0, 0] | -15 | 12 | 0 |
| 10 | [0, 1, 0, 1] | -1 | -14 | 2 |
| 10 | [0, 1, 0, 1] | -18 | -4 | -2 |
| 10 | [0, 1, 0, 1] | -51 | 6 | 0 |
| 10 | [0, 1, 0, 1] | -1 | 16 | 2 |
| 10 | [0, 1, 0, 2] | -1 | -10 | -2 |
| 10 | [0, 1, 0, 2] | -30 | 0 | 0 |
| 10 | [0, 1, 0, 2] | -1 | 10 | 2 |
| 10 | [0, 1, 0, 3] | -3 | -6 | 0 |
| 10 | [0, 1, 0, 3] | -1 | 4 | 2 |
| 10 | [0, 1, 1, 0] | -3 | -10 | -2 |
| 10 | [0, 1, 1, 0] | -59 | 0 | 0 |
| 10 | [0, 1, 1, 0] | -3 | 10 | 2 |
| 10 | [0, 1, 1, 1] | -17 | -6 | 0 |
| 10 | [0, 1, 1, 1] | -3 | 4 | 2 |
| 10 | [0, 1, 1, 2] | -1 | -2 | 2 |
| 10 | [0, 2, 0, 0] | -24 | -6 | 0 |
| 10 | [0, 2, 0, 0] | -3 | 4 | 2 |
| 10 | [0, 2, 0, 1] | -2 | -12 | 0 |
| 10 | [0, 2, 0, 1] | -2 | -2 | 2 |
| 10 | [0, 3, 0, 0] | -1 | 6 | 0 |
| 10 | [1, 0, 0, 0] | -2 | -14 | 2 |
| 10 | [1, 0, 0, 0] | -18 | -4 | -2 |
| 10 | [1, 0, 0, 0] | -37 | 6 | 0 |
| 10 | [1, 0, 0, 0] | -1 | 16 | 2 |
| 10 | [1, 0, 0, 1] | -8 | -10 | -2 |
| 10 | [1, 0, 0, 1] | -74 | 0 | 0 |
| 10 | [1, 0, 0, 1] | -8 | 10 | 2 |
| 10 | [1, 0, 0, 2] | -20 | -6 | 0 |
| 10 | [1, 0, 0, 2] | -8 | 4 | 2 |
| 10 | [1, 0, 0, 3] | -2 | -2 | 2 |
| 10 | [1, 0, 1, 0] | -1 | -16 | -2 |
| 10 | [1, 0, 1, 0] | -51 | -6 | 0 |
| 10 | [1, 0, 1, 0] | -18 | 4 | 2 |
| 10 | [1, 0, 1, 0] | -1 | 14 | -2 |
| 10 | [1, 0, 1, 1] | -4 | -12 | 0 |
| 10 | [1, 0, 1, 1] | -10 | -2 | 2 |
| 10 | [1, 0, 1, 1] | -3 | 8 | -2 |
| 10 | [1, 0, 1, 2] | -1 | -8 | 2 |
| 10 | [1, 0, 1, 2] | -1 | 2 | -2 |
| 10 | [1, 0, 2, 0] | -2 | 2 | -2 |
| 10 | [1, 0, 2, 0] | -2 | 12 | 0 |
| 10 | [1, 0, 2, 1] | -2 | 6 | 0 |
| 10 | [1, 1, 0, 0] | -15 | -12 | 0 |
| 10 | [1, 1, 0, 0] | -14 | -2 | 2 |
| 10 | [1, 1, 0, 0] | -7 | 8 | -2 |
| 10 | [1, 1, 0, 1] | -3 | -8 | 2 |
| 10 | [1, 1, 0, 1] | -10 | 2 | -2 |
| 10 | [1, 1, 0, 1] | -4 | 12 | 0 |
| 10 | [1, 1, 0, 2] | -1 | -4 | -2 |
| 10 | [1, 1, 0, 2] | -4 | 6 | 0 |
| 10 | [1, 1, 1, 0] | -3 | -4 | -2 |
| 10 | [1, 1, 1, 0] | -17 | 6 | 0 |
| 10 | [1, 1, 1, 1] | -6 | 0 | 0 |
| 10 | [1, 2, 0, 0] | -15 | 0 | 0 |
| 10 | [1, 2, 0, 1] | -2 | -6 | 0 |
| 10 | [2, 0, 0, 0] | -1 | -18 | 0 |
| 10 | [2, 0, 0, 0] | -3 | -8 | 2 |
| 10 | [2, 0, 0, 0] | -12 | 2 | -2 |
| 10 | [2, 0, 0, 0] | -4 | 12 | 0 |
| 10 | [2, 0, 0, 1] | -8 | -4 | -2 |
| 10 | [2, 0, 0, 1] | -20 | 6 | 0 |
| 10 | [2, 0, 0, 2] | -8 | 0 | 0 |
| 10 | [2, 0, 1, 0] | -1 | -10 | -2 |
| 10 | [2, 0, 1, 0] | -30 | 0 | 0 |
| 10 | [2, 0, 1, 0] | -1 | 10 | 2 |
| 10 | [2, 0, 1, 1] | -4 | -6 | 0 |
| 10 | [2, 0, 1, 1] | -1 | 4 | 2 |
| 10 | [2, 1, 0, 0] | -16 | -6 | 0 |
| 10 | [2, 1, 0, 0] | -3 | 4 | 2 |
| 10 | [2, 1, 0, 1] | -1 | -2 | 2 |
| 10 | [2, 1, 0, 1] | -1 | 8 | -2 |
| 10 | [2, 1, 1, 0] | -1 | 2 | -2 |
| 10 | [2, 2, 0, 0] | -1 | 6 | 0 |
| 10 | [3, 0, 0, 0] | -2 | -12 | 0 |
| 10 | [3, 0, 0, 0] | -2 | -2 | 2 |
| 10 | [3, 0, 0, 0] | -1 | 8 | -2 |
| 10 | [3, 0, 0, 1] | -2 | 2 | -2 |
| 10 | [3, 0, 1, 0] | -1 | -4 | -2 |
| 10 | [3, 0, 1, 0] | -3 | 6 | 0 |
| 10 | [3, 1, 0, 0] | -4 | 0 | 0 |
| 10 | [4, 0, 0, 0] | -1 | -6 | 0 |

## Appendix J. Master check catalogue

| ID | stage | claim | status | evidence |
|---|---|---|---|---|
| NB-HWG-ALL-MULTIPLICITIES-ARE-INTEGERS-A660DE | hwg | all multiplicities are integers | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-ALL-MULTIPLICITIES-ARE-NONNEGATIVE-998368 | hwg | all multiplicities are nonnegative | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-ALL-PASSED-149BD5 | hwg | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-CONSTANT-COEFFICIENT-IS-ONE-7EDAD8 | hwg | constant coefficient is one | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-NO-TERMS-ABOVE-REQUESTED-DEGREE-652FED | hwg | no terms above requested degree | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-PE-EQUALS-RATIONAL-PRODUCT-0FDDCA | hwg | pe equals rational product | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-TRUNCATION-STABILITY-0E4777 | hwg | truncation stability | PASS | generated/su3_nf5_k3o2_infinite/order_10/hwg_expansion.json |
| NB-HWG-ALL-MULTIPLICITIES-ARE-INTEGERS-A660DE | hwg | all multiplicities are integers | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-HWG-ALL-MULTIPLICITIES-ARE-NONNEGATIVE-998368 | hwg | all multiplicities are nonnegative | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-HWG-ALL-PASSED-149BD5 | hwg | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-HWG-CONSTANT-COEFFICIENT-IS-ONE-7EDAD8 | hwg | constant coefficient is one | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-HWG-NO-TERMS-ABOVE-REQUESTED-DEGREE-652FED | hwg | no terms above requested degree | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-HWG-PE-EQUALS-RATIONAL-PRODUCT-0FDDCA | hwg | pe equals rational product | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-HWG-TRUNCATION-STABILITY-0E4777 | hwg | truncation stability | PASS | generated/su3_nf5_k3o2_infinite/order_10/checks.json |
| NB-CHAR-ALL-PASSED-7B5F57 | characters | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-DIMENSIONS-NONNEGATIVE-INTEGERS-03E5B0 | characters | dimensions nonnegative integers | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-LEADING-COEFFICIENTS-B23D64 | characters | leading coefficients | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-ONE-IRREP-PER-HWG-MONOMIAL-BEFORE-COMBINING-BF4BB2 | characters | one irrep per hwg monomial before combining | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-Q-CHARGES-PRESERVED-194BCD | characters | q charges preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-Q-EQUALS-ONE-MATCHES-UNREFINED-357DAC | characters | q equals one matches unrefined | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-REPRESENTATION-MULTIPLICITIES-NONNEGATIVE-INTEGERS-2693EB | characters | representation multiplicities nonnegative integers | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-CHAR-T-DEGREES-PRESERVED-6C26A3 | characters | t degrees preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/character_checks.json |
| NB-PL-ALL-FINAL-COEFFICIENTS-INTEGRAL-EFBA6F | plethystic-log | all final coefficients integral | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-ALL-PASSED-5FA8BC | plethystic-log | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-DEGREE-2-INDEPENDENT-VALUE-9F9007 | plethystic-log | degree 2 independent value | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-DEGREE-3-INDEPENDENT-VALUE-2B1E38 | plethystic-log | degree 3 independent value | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-DEGREE-4-INDEPENDENT-VALUE-24017E | plethystic-log | degree 4 independent value | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-DEGREES-TRUNCATED-E37BE2 | plethystic-log | degrees truncated | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-DIRECT-SCALAR-MATCHES-REFINED-UNREFINEMENT-18C4A8 | plethystic-log | direct scalar matches refined unrefinement | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-PL-NEGATIVE-COEFFICIENTS-RETAINED-8CE499 | plethystic-log | negative coefficients retained | PASS | generated/su3_nf5_k3o2_infinite/order_10/plethystic_logarithm_checks.json |
| NB-RECON-ALL-PASSED-090256 | reconstruction | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-DIFFERENCE-IS-EMPTY-D41C49 | reconstruction | difference is empty | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-INDEPENDENT-SCALAR-PE-EQUAL-89BBC1 | reconstruction | independent scalar pe equal | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-MULTIPLICITIES-INTEGRAL-319D5D | reconstruction | multiplicities integral | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-MULTIPLICITIES-NONNEGATIVE-FEE6C7 | reconstruction | multiplicities nonnegative | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-ORDER-STABILITY-58D959 | reconstruction | order stability | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-Q-REFINED-DIMENSIONS-EQUAL-F01E6D | reconstruction | q refined dimensions equal | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-REFINED-CHARACTER-SERIES-EQUAL-028BF2 | reconstruction | refined character series equal | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-RECON-UNREFINED-DIMENSIONS-EQUAL-92E74F | reconstruction | unrefined dimensions equal | PASS | generated/su3_nf5_k3o2_infinite/order_10/reconstruction_checks.json |
| NB-OPER-ALL-PASSED-7F0314 | operator-analysis | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json |
| NB-OPER-DEFICIT-EQUALS-NEGATIVE-PL-CONTENT-5087F8 | operator-analysis | deficit equals negative pl content | PASS | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json |
| NB-OPER-DIMENSIONS-EXACT-FD7A9A | operator-analysis | dimensions exact | PASS | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json |
| NB-OPER-FIRST-NEGATIVE-DEGREE-EXISTS-14B9F4 | operator-analysis | first negative degree exists | PASS | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json |
| NB-OPER-RELATION-DEGREE-NOT-MIXED-17C876 | operator-analysis | relation degree not mixed | PASS | generated/su3_nf5_k3o2_infinite/order_10/operator_content_checks.json |
| NB-BRANCH-ALL-PARENT-DIMENSIONS-PRESERVED-A7F1EB | branching | all parent dimensions preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-ALL-PASSED-A49E09 | branching | all passed | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-CANDIDATE-GENERATOR-DIMENSION-82D7F6 | branching | candidate generator dimension | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-CHARACTER-UNREFINEMENT-PRESERVED-750FA6 | branching | character unrefinement preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-FIRST-RELATION-DIMENSION-05AA5A | branching | first relation dimension | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-ORDINARY-MULTIPLICITIES-NONNEGATIVE-INTEGERS-C445D1 | branching | ordinary multiplicities nonnegative integers | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-PHYSICAL-CHARGE-MAP-ASSUMED-82B2DF | branching | physical charge map assumed | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-PLETHYSTIC-LOG-UNREFINEMENT-PRESERVED-9109E6 | branching | plethystic log unrefinement preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-Q-CHARGES-PRESERVED-04AB6C | branching | q charges preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-RAW-CHARGE-BASIS-ONLY-ADDAB0 | branching | raw charge basis only | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-RECONSTRUCTED-CHARACTER-BRANCHING-EQUAL-A30CE6 | branching | reconstructed character branching equal | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-SIGNED-VIRTUAL-MULTIPLICITIES-RETAINED-06925D | branching | signed virtual multiplicities retained | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-T-DEGREES-PRESERVED-EF8BCC | branching | t degrees preserved | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-X-CHARGES-ARE-EXACT-INTEGERS-E454CE | branching | x charges are exact integers | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-3-0-66CB7D | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-2-1-BB8D7C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-2-2-8D284F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-2-0-F537FF | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-2-0-2-1-2032EB | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-1-2-73AF97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-0-0-3-DE5AA8 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-1-0-1-3-9DB37C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-4-0-0-0-4-1B1AC7 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-5-0-0-0-5-D2F865 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-3-0-1-0-D2C728 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-2-0-0-1-66D755 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-2-0-0-2-F4B444 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-0-0-3-DE5AA8 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-2-0-7B9D8A | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-1-2-1744B7 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-1-0-AA1767 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-0-2-F2D8C0 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-2-1-BB8D7C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-2-0-F537FF | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-1-2-73AF97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-0-0-3-DE5AA8 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-4-0-0-0-4-1B1AC7 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-2-0-0-1-66D755 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-3-0-BFC810 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-2-0-7B9D8A | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-2-1-40E138 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-1-2-1744B7 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-0-1-3-F1B4B4 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-1-0-AA1767 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-2-0-1-1-1115E7 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-0-2-F2D8C0 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-1-0-0-3-76EAB4 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-3-0-0-0-CD61F4 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-2-97E2BE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-2-0-F4F110 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-2-9EED6D | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-1-0-28C4F9 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-1-0-47C4FB | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-0-0-1-2750BE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-2-2-628F6E | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-3-0-BFC810 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-0-3-8AEB48 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-1-1-78228C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-2-0-0-01199B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-2-20C623 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-3-0-0-0-CD61F4 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-2-0-1-45357C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-1-1-8B4C68 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-1-0-0-B4EEF2 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-2-7AE2A5 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-1-0-B41EA8 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-2-0-0-0-C2339F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-1-0-0-D8BC3C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-2-001385 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-1-0-1-F82C0F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-2-0-0-0-726DE1 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-3-C49459 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-1-0-1-51D7B4 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-0-0-BA2108 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-2-0-0-01199B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-0-0-BA2108 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-2-97E2BE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-1-1-78228C | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-2-0-0-01199B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-2-20C623 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-1-0-56474F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-1-0-0-B4EEF2 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-1-0-B41EA8 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-2-BE57FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-2-1-0-F892A5 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-1-0-1-F82C0F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-3-C49459 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-1-1-5FDA97 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-0-0C83FE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-1-0-1-51D7B4 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-1-0-0-0-BA2108 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-4-0-0-0-0-A4900A | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-4-CAC6D9 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-2-97E2BE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-1-0-1-8631FC | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-2-0-0-D4866F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-0-2-9EED6D | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-1-1-0-28C4F9 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-1-0-0-1-B0332B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-2-0-0-0-0-E10163 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-3-0-0-0-1-2750BE | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-1-0-423E24 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-1-0-0-0-DFB76F | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-0-0-0-0-0-FCD731 | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |
| NB-BRANCH-DIMENSION-1-0-0-0-1-057E3B | branching | parent and child dimensions agree | PASS | generated/su3_nf5_k3o2_infinite/order_10/manifest_branching/branching_checks.json |